# 🚀 Cocopila Financial Data Agent Pipeline (Kaggle Bootstrap)

Notebook này chứa **toàn bộ mã nguồn Agent Pipeline & Thiết lập môi trường Kaggle** bao gồm:
0. **Kaggle Clone & Workspace Setup**: Clone mã nguồn từ Public Repo vào `/kaggle/working/r2AI_2026`.
1. **Môi trường & Phụ thuộc**: Cài đặt Ollama Linux Binary, Python dependencies (`qdrant-client`, `sentence-transformers`, `rank-bm25`, `langgraph`, `thefuzz`, ...) & pull model `qwen2.5-coder:1.5b`.
2. **System Configuration, Provider & Utilities**: Cấu hình hệ thống, kết nối LLM, JSON repair utility.
3. **Prompts Mẫu (YAML Prompt Templates)**: Query Parser, Code Generator & Reflection Debugging.
4. **Agent State Definition**: Shared State Dictionary dùng trong LangGraph.
5. **Toàn bộ 5 Agent Pipeline Nodes**:
   - **Node 1: Query Parser** (Phân tích câu hỏi tài chính thành JSON cấu trúc & trích xuất khái niệm cốt lõi)
   - **Node 2: Data Discovery** (Tìm kiếm bảng dữ liệu phù hợp với Search Engine & DataRegistry)
   - **Node 3: Schema Mapper** (Ánh xạ tiêu chí phụ sang tên cột thực tế trong CSV)
   - **Node 4: Code Generator & Reflection** (Sinh mã Python/Pandas trích xuất/tính toán/so sánh & tự động sửa lỗi)
   - **Node 5: AST Sandbox & Executor** (Thực thi mã Python an toàn trong Sandbox AST)
6. **Workflow StateGraph & Conditional Edge Routing**: Khởi tạo LangGraph app với vòng lặp Reflection Loop.
7. **Kiểm thử trực tiếp trên 10 câu hỏi ngẫu nhiên (seed=42)**

## 🛠️ Section 0: Kaggle Workspace Setup & Code Cloning

In [3]:
# 0. Clone mã nguồn dự án vào thư mục /kaggle/working/r2AI_2026
import os
import sys
import shutil
import subprocess
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "r2AI_2026"
REPO_URL = "https://github.com/Djuybu/r2AI_2026.git"

# Kiểm tra Token nếu repo là Private (Lấy từ Kaggle Secrets hoặc biến môi trường)
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    pass

if GITHUB_TOKEN and "github.com" in REPO_URL:
    auth_repo_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    auth_repo_url = REPO_URL

if os.path.exists("/kaggle/working"):
    print("🚀 Phát hiện môi trường Kaggle! Đang chuẩn bị thư mục làm việc...")
    if not REPO_DIR.exists():
        print(f"📥 Đang clone repository từ {REPO_URL} vào {REPO_DIR}...")
        res = subprocess.run(["git", "clone", auth_repo_url, str(REPO_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"⚠️ Lỗi Git Clone: {res.stderr.strip()}")
            # Dò tìm mã nguồn trong /kaggle/input làm phương án dự phòng
            dataset_candidates = list(Path("/kaggle/input").glob("**/r2AI_2026")) if Path("/kaggle/input").exists() else []
            if dataset_candidates:
                src_path = dataset_candidates[0]
                print(f"📦 Tìm thấy mã nguồn trong Kaggle Input Dataset: {src_path}. Đang sao chép sang {REPO_DIR}...")
                shutil.copytree(src_path, REPO_DIR, dirs_exist_ok=True)

    if REPO_DIR.exists():
        os.chdir(str(REPO_DIR))
        if str(REPO_DIR) not in sys.path:
            sys.path.insert(0, str(REPO_DIR))
        print(f"✅ Đã chuyển thư mục làm việc: {os.getcwd()}")
    else:
        print(f"⚠️ Không tìm thấy thư mục {REPO_DIR}. Tiếp tục với thư mục làm việc mặc định: {os.getcwd()}")
else:
    print(f"💻 Đang chạy trên môi trường Local: {os.getcwd()}")

🚀 Phát hiện môi trường Kaggle! Đang chuẩn bị thư mục làm việc...
📥 Đang clone repository từ https://github.com/Djuybu/r2AI_2026.git vào /kaggle/working/r2AI_2026...
✅ Đã chuyển thư mục làm việc: /kaggle/working/r2AI_2026


In [4]:
# Cài đặt các công cụ hệ thống hỗ trợ
!sudo apt-get update -y
!sudo apt install lshw -y
!sudo apt-get install zstd -y


Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [112 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]  
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http:/

## 📥 Section 0.1: Installing Ollama CLI & Dependencies on Kaggle

In [5]:
# 0.1 Cài đặt Ollama CLI trên Linux kernel của Kaggle (nếu chưa có)
print("📥 Đang kiểm tra / cài đặt Ollama CLI trên Kaggle Linux...")
!curl -fsSL https://ollama.com/install.sh | sh

📥 Đang kiểm tra / cài đặt Ollama CLI trên Kaggle Linux...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                                       0.1%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
# 1. Cài đặt đầy đủ các gói phụ thuộc dự án (Bao gồm LangGraph, Qdrant Client, Sentence Transformers, BM25, ...)
print("📥 Đang cài đặt Python dependencies cho Agent Pipeline & RAG Search Engine...")
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    pyyaml>=6.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    thefuzz>=0.22.0 \
    qdrant-client \
    sentence-transformers \
    rank-bm25

📥 Đang cài đặt Python dependencies cho Agent Pipeline & RAG Search Engine...


In [7]:
# 2. Khởi động Ollama Server chạy nền & Tải model
import subprocess
import time
import requests
import os

print("🚀 Đang khởi động Ollama Server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Chờ Ollama Server khởi động...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/")
        if r.status_code == 200:
            print("✅ Ollama Server đã sẵn sàng tại port 11434!")
            break
    except Exception:
        time.sleep(1)
else:
    print("❌ Lỗi: Ollama Server không thể khởi động.")

MODEL_NAME = "qwen2.5-coder:1.5b"
print(f"📥 Đang tải mô hình {MODEL_NAME} từ Ollama registry...")
subprocess.run(["ollama", "pull", MODEL_NAME])
print(f"✅ Đã tải thành công mô hình {MODEL_NAME}!")

os.environ["MODEL_NAME"] = MODEL_NAME
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"

🚀 Đang khởi động Ollama Server...
⏳ Chờ Ollama Server khởi động...
✅ Ollama Server đã sẵn sàng tại port 11434!
📥 Đang tải mô hình qwen2.5-coder:1.5b từ Ollama registry...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 29d8c98fa6b0:   0% ▕                  ▏ 2.4 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:   7% ▕█                 ▏  72 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  12% ▕██                ▏ 121 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  21% ▕███               ▏ 203 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  30% ▕█████             ▏ 292 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  34% ▕██████            ▏ 333 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  43% ▕███████           ▏ 424 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  52% ▕█████████         ▏ 508 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  56% ▕██████████        ▏ 549 MB/986 MB                  pulling manif

✅ Đã tải thành công mô hình qwen2.5-coder:1.5b!


pulling manifest 
pulling 29d8c98fa6b0: 100% ▕█████████████████ ▏ 985 MB/986 MB  715 MB/s      0s
verifying sha256 digest 
writing manifest 
success 


## ⚙️ Section 1: System Configuration, LLM Provider & Utilities

In [8]:
import os
import sys
import json
import logging
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Optional, Literal, TypedDict
from json_repair import repair_json
from langchain_openai import ChatOpenAI
from langchain_core.language_models.chat_models import BaseChatModel

def is_kaggle_environment() -> bool:
    """Check if execution environment is Kaggle."""
    return os.path.exists("/kaggle/working")

# ==============================================================================
# Dual Output Logger: Ghi toàn bộ kết quả in ra cả Console và file .txt
# ==============================================================================
class DualOutputLogger:
    """Redirects sys.stdout so that all outputs are printed to console AND saved into a .txt log file."""
    def __init__(self, log_filepath: str = "pipeline_execution.txt"):
        self.terminal = sys.__stdout__
        self.log_filepath = Path(log_filepath)
        self.log_filepath.parent.mkdir(parents=True, exist_ok=True)
        self.log_file = open(self.log_filepath, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def close(self):
        if not self.log_file.closed:
            self.log_file.close()

LOG_FILE_PATH = Path("/kaggle/working/pipeline_execution.txt") if is_kaggle_environment() else Path("./pipeline_execution.txt")
if not isinstance(sys.stdout, DualOutputLogger):
    sys.stdout = DualOutputLogger(str(LOG_FILE_PATH))

print(f"📝 Đã kích hoạt ghi log tự động ra file .txt: {LOG_FILE_PATH.resolve()}")

@dataclass
class Config:
    """System configuration parameters."""
    MODEL_NAME: str = os.getenv("MODEL_NAME", "qwen2.5-coder:1.5b")
    LLM_API_BASE: str = os.getenv("LLM_API_BASE", "http://localhost:11434/v1")
    LLM_API_KEY: str = os.getenv("LLM_API_KEY", "ollama")
    TEMPERATURE: float = float(os.getenv("TEMPERATURE", "0.0"))
    MAX_TOKENS: int = int(os.getenv("MAX_TOKENS", "1024"))
    BASE_DIR: Path = Path("/kaggle/working/r2AI_2026/pipeline") if is_kaggle_environment() else Path.cwd()
    DATA_DIR: Path = Path(os.getenv("DATA_DIR", "/kaggle/working/r2AI_2026/pipeline/data"))
    MAX_RETRIES: int = int(os.getenv("MAX_RETRIES", "3"))

config = Config()

def get_llm(
    cfg: Optional[Config] = None,
    temperature: Optional[float] = None,
    max_tokens: Optional[int] = None,
    timeout: Optional[int] = 30,
    **kwargs,
) -> BaseChatModel:
    """Instantiate ChatOpenAI connected to local LLM endpoint safely supporting timeout & kwargs."""
    cfg = cfg or config
    temp = temperature if temperature is not None else cfg.TEMPERATURE
    tokens = max_tokens if max_tokens is not None else cfg.MAX_TOKENS
    
    llm_kwargs = {
        "model": cfg.MODEL_NAME,
        "base_url": cfg.LLM_API_BASE,
        "api_key": cfg.LLM_API_KEY,
        "temperature": temp,
        "max_tokens": tokens,
        "streaming": False,
        **kwargs,
    }
    if timeout is not None:
        llm_kwargs["timeout"] = timeout
        
    return ChatOpenAI(**llm_kwargs)

def safe_parse_json(content: str) -> Dict[str, Any]:
    """Parse JSON string with automatic repair fallback."""
    if not content or not content.strip():
        return {}
    cleaned = content.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    try:
        repaired = repair_json(cleaned)
        return json.loads(repaired)
    except Exception as exc:
        print(f"⚠️ Safe JSON parse failed: {exc}")
        return {}

print("✅ System configuration, Logging to .txt & LLM Provider utilities loaded!")

✅ System configuration & LLM Provider utilities loaded!


## 📝 Section 2: Prompts Mẫu (Prompt Templates)
Định nghĩa các Prompt mẫu cho Query Parser, Code Generator và Reflection Debugging Loop.

In [9]:
PROMPT_QUERY_PARSER = {
    "system_prompt": """You MUST output ONLY a valid JSON object with the exact keys: {"ticker": "string", "year": "string", "metric": "string"}.
Your output will be parsed directly by `json.loads()`. Any markdown code blocks (like ```json) or explanations WILL CRASH THE SYSTEM. Output raw JSON only.

CRITICAL INSTRUCTION: The JSON examples provided above are STRICTLY for formatting demonstration. DO NOT copy the values (ticker, year, metric) from the examples. You MUST read the actual [USER_QUERY] provided and dynamically extract the REAL ticker, REAL year, and REAL metric requested by the user.

## DETAILED INSTRUCTIONS:
1. "ticker": Map the target company or bank name in the query to its exact 3-5 letter uppercase ticker symbol using the stock mapping (code_stock.csv). (e.g., "Vietjet" -> "VJC", "Ngân hàng TMCP Sài Gòn Thương Tín" -> "STB", "FPT" -> "FPT", "Tập đoàn Vingroup" -> "VIC"). If no company is mentioned or if the company is unknown, output an empty string "". DO NOT output null or None!

2. "year": Extract the target year or list of years from the query as a string (e.g., "2023" or "2021, 2022, 2023"). If no year is found, output an empty string "". DO NOT output null or None!

3. "metric": Extract the exact financial metric string verbatim from the query (e.g., "Lãi tiền gửi", "Chi phí khác", "Doanh thu thuần", "Lãi vay phải trả"). Do NOT copy metrics from examples!

OUTPUT FORMAT: Output ONLY raw valid JSON matching {"ticker": "string", "year": "string", "metric": "string"}.""",
    "user_prompt_template": """Câu hỏi: {user_query}

Ví dụ mẫu:
{few_shot_examples}

Hãy phân tích câu hỏi trên và trả về JSON:""",
    "few_shot_examples": [
        {
            "user_query": "Lãi tiền gửi năm 2021 của Vietjet (VJC) là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "VJC", "year": "2021", "metric": "Lãi tiền gửi"}'
        },
        {
            "user_query": "Chi phí lương và các khoản khác theo lương của công ty mẹ CTCP Chứng khoán FPT trong năm 2021 là bao nhiêu tỷ đồng?",
            "parsed_output": '{"ticker": "FPT", "year": "2021", "metric": "Chi phí lương và các khoản khác theo lương"}'
        },
        {
            "user_query": "Lãi vay phải trả của CTCP Tập đoàn Đức Long Gia Lai (DLG) cuối năm 2023 là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "DLG", "year": "2023", "metric": "Lãi vay phải trả"}'
        },
        {
            "user_query": "Doanh thu hoạt động tài chính năm 2020 của Ngân hàng TMCP Sài Gòn Thương Tín (STB)",
            "parsed_output": '{"ticker": "STB", "year": "2020", "metric": "Doanh thu hoạt động tài chính"}'
        },
        {
            "user_query": "Vốn chủ sở hữu của FIT là bao nhiêu tỷ đồng vào ngày 31/12/2015?",
            "parsed_output": '{"ticker": "FIT", "year": "2015", "metric": "Vốn chủ sở hữu"}'
        },
        {
            "user_query": "Tốc độ tăng trưởng % tổng tiền và các khoản tương đương tiền từ năm 2019 đến năm 2021 của VIC",
            "parsed_output": '{"ticker": "VIC", "year": "2019, 2020, 2021", "metric": "tổng tiền và các khoản tương đương tiền"}'
        }
    ]
}

PROMPT_SCHEMA_MAPPER = {
    "system_prompt": """Bạn là chuyên gia phân tích dữ liệu tài chính Việt Nam.
Cho danh sách tên cột giá trị trong bảng báo cáo tài chính, hãy mô tả ngắn gọn (1 câu tiếng Việt) ý nghĩa từng cột.
Trả về ONLY một JSON array với format:
[{"column_name": "tên cột", "column_description": "mô tả ngắn gọn"}]
KHÔNG giải thích, KHÔNG bọc markdown. Output raw JSON only.""",
    "user_prompt_template": """Bảng: {table_name}
Danh sách cột giá trị: {columns}
Hãy mô tả ý nghĩa từng cột giá trị trong bảng báo cáo tài chính này:"""
}

PROMPT_CODE_GENERATOR = {
    "system_prompt": """Bạn là một chuyên gia lập trình Python & Pandas Data Analysis.
Nhiệm vụ của bạn là sinh ra đoạn mã Python để truy vấn dữ liệu từ bảng báo cáo tài chính.

<BANNED_SYNTAX>
- You are FORBIDDEN from writing `result = 0.0` or `result = 0` or any fallback value at the end of the script to bypass errors.
- You are FORBIDDEN from using `.iloc[0]` or `.values[0]` outside of an `if not filtered_df.empty:` block.
- You are FORBIDDEN from using `str.contains()` without explicitly passing `regex=False`.
- You are FORBIDDEN from redefining `clean_val` or `extract_value`. They are pre-injected into your execution scope.
</BANNED_SYNTAX>

CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table"). You MUST let the script crash if the exact metric is not found!
CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.

QUY TẮC BẮT BUỘC (TUYỆT ĐỐI TUÂN THỦ):
1. Dữ liệu nằm trong file CSV. Đọc file bằng `pd.read_csv(file_path)`.
2. Cấu trúc bảng: Cột nhãn (label_column) chứa tên chỉ tiêu. Cột giá trị (value_column) chứa số liệu.
3. BẮT BUỘC dùng `regex=False` khi sử dụng `str.contains` để tránh lỗi regex với các chỉ tiêu có dấu ngoặc tròn (ví dụ: `df['CHỈ TIÊU'].astype(str).str.contains('Lợi nhuận sau thuế', case=False, na=False, regex=False)`).
4. BẮT BUỘC sử dụng mẫu code an toàn:
   filtered_df = df[df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)]
   if not filtered_df.empty:
       result = extract_value(filtered_df.iloc[0], '{value_col}')
   else:
       raise ValueError("Metric not found in table")
5. QUY TẮC SỬ DỤNG SCHEMA:
   - Nếu bảng có NHIỀU HƠN MỘT CỘT GIÁ TRỊ, so sánh chỉ tiêu cần tìm với tên và mô tả của từng cột trong SCHEMA để chọn đúng cột giá trị phù hợp nhất.
   - Nếu dữ liệu cần tìm là một SECTION (danh mục) trong SCHEMA:
     + Ưu tiên lấy trực tiếp `total_value` của section đó nếu có.
     + Nếu `total_value` không có hoặc rỗng, thực hiện tính tổng các hàng nằm trong `range` [start, end] của section đó trên cột giá trị: `df.iloc[start:end+1]`.
6. XỬ LÝ KHI KHÔNG TÌM THẤY DỮ LIỆU / DỮ LIỆU RỖNG:
   - Khi không lọc thấy dòng hoặc extract_value không tìm thấy giá trị, BẮT BUỘC raise `ValueError("Metric not found in table")` để kích hoạt Reflection Loop.
7. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.
8. CHỈ trả về code Python thuần túy. KHÔNG giải thích, KHÔNG bọc markdown.
""",
    "goal_descriptions": {
        "trich_xuat": "TRÍCH XUẤT giá trị cụ thể",
        "tinh_tong": "TÍNH TỔNG (tìm dòng Tổng/Cộng trước, hoặc tính tổng section theo range)",
        "so_sanh": "SO SÁNH giá trị giữa nhiều năm/công ty"
    },
    "goal_instructions": {
        "trich_xuat": """HƯỚNG DẪN CỤ THỂ (TRÍCH XUẤT):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Nếu nội dung cần tìm là một section: kiểm tra nếu có total_value thì lấy total_value, nếu không thì cộng các dòng trong range [start:end+1].
3. Ngược lại, truy vấn hàng ở cột '{label_col}' với `df['{label_col}'].astype(str).str.contains('{noi_dung}', case=False, na=False, regex=False)`.
4. Dùng kiểm tra `if not filtered_df.empty:` để lấy giá trị qua `extract_value(filtered_df.iloc[0], '{value_col}')`. Nếu rỗng → raise ValueError("Metric not found in table").
5. Gán vào biến result.""",
        "tinh_tong": """HƯỚNG DẪN CỤ THỂ (TÍNH TỔNG):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Nếu nội dung cần tìm là một section: kiểm tra nếu có total_value thì lấy total_value, nếu không thì lấy `sub_df = df.iloc[start:end+1]` và tính tổng các dòng qua `clean_val`.
3. Ngược lại, tìm dòng có chứa 'Tổng' hoặc 'Cộng' hoặc tên chỉ tiêu '{noi_dung}' ở cột '{label_col}' bằng `df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)`.
4. Kiểm tra `if not filtered_df.empty:` để lấy giá trị qua `extract_value(filtered_df.iloc[0], '{value_col}')`. Nếu không tìm thấy, cộng các dòng con liên quan. Nếu vẫn rỗng → raise ValueError("Metric not found in table").
5. Gán kết quả vào result.""",
        "so_sanh": """HƯỚNG DẪN CỤ THỂ (SO SÁNH NĂM / TÍNH TỐC ĐỘ TĂNG TRƯỞNG):
1. Đọc từng file CSV cho từng năm (ví dụ file_path_2019, file_path_2020, file_path_2021...).
2. Truy vấn hàng ở cột '{label_col}' với `str.contains(..., case=False, na=False, regex=False)` và lấy giá trị từng năm qua `extract_value(filtered_df.iloc[0], '{value_col}')`. Nếu rỗng → raise ValueError("Metric not found in table").
3. Tính tốc độ tăng trưởng phần trăm (%) giữa năm đầu và năm cuối: `growth_rate = ((val_last - val_first) / val_first) * 100`.
4. Gán kết quả vào `result`."""
    },
    "user_prompt_template": """Yêu cầu người dùng: {user_query}

MỤC TIÊU: {muc_tieu_desc}
NỘI DUNG cần tìm (ở cột label): '{noi_dung}'
Công ty: {ten_cong_ty}
Năm: {so_nam}
Tiêu chí phụ: {tieu_chi_phu}

DỮ LIỆU CÓ SẴN:
{files_context}

CỘT QUAN TRỌNG:
- Cột nhãn (chứa tên chỉ tiêu): '{label_col}'
- Cột giá trị: '{value_col}'

BIẾN ĐƯỜNG DẪN FILE:
{paths_str}

{goal_instruction}

<BANNED_SYNTAX>
- You are FORBIDDEN from writing `result = 0.0` or `result = 0` or any fallback value at the end of the script to bypass errors.
- You are FORBIDDEN from using `.iloc[0]` or `.values[0]` outside of an `if not filtered_df.empty:` block.
- You are FORBIDDEN from using `str.contains()` without explicitly passing `regex=False`.
</BANNED_SYNTAX>

🚨 BẮT BUỘC KHÔNG ĐƯỢC VI PHẠM:
1. CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table").
2. CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.
3. Đọc file bằng pd.read_csv(file_path...).
4. Dùng mẫu safe pattern với `regex=False`: `filtered_df = df[df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)]`
5. BẮT BUỘC dùng `if not filtered_df.empty:` trước khi lấy `.iloc[0]`. Nếu rỗng, BẮT BUỘC `raise ValueError("Metric not found in table")`. KHÔNG ĐƯỢC gán result = 0.0!
6. KHÔNG ĐƯỢC filter theo `df['Ma_Doanh_Nghiep'] == ...` vì dữ liệu đã đúng công ty.
7. CHỈ ĐƯỢC SỬ DỤNG CÁC BIẾN ĐƯỜNG DẪN FILE ĐÃ ĐƯỢC ĐỊNH NGHĨA Ở TRÊN:
{paths_str}
8. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.""",
    "few_shot_examples": [
        {
            "user_query": "Doanh thu thuần năm 2023 của FPT",
            "file_path": "data/FPT_2023.csv",
            "column_mapping": '{"label_column": "CHỈ TIÊU", "value_column": "Năm nay"}',
            "generated_code": """import pandas as pd

df = pd.read_csv(file_path)
filtered_df = df[df['CHỈ TIÊU'].astype(str).str.contains('Doanh thu thuần', case=False, na=False, regex=False)]
if not filtered_df.empty:
    result = extract_value(filtered_df.iloc[0], 'Năm nay')
else:
    raise ValueError("Metric not found in table")"""
        }
    ]
}

PROMPT_REFLECTION = {
    "system_prompt": """Bạn là một chuyên gia lập trình Python & Pandas Data Analysis.
Nhiệm vụ của bạn là SỬA LỖI mã nguồn Python đã sinh ra ở bước trước dựa trên thông báo lỗi (Traceback) và mẫu dữ liệu thực tế từ bảng báo cáo tài chính.

<BANNED_SYNTAX>
- You are FORBIDDEN from writing `result = 0.0` or `result = 0` or any fallback value at the end of the script to bypass errors.
- You are FORBIDDEN from using `.iloc[0]` or `.values[0]` outside of an `if not filtered_df.empty:` block.
- You are FORBIDDEN from using `str.contains()` without explicitly passing `regex=False`.
</BANNED_SYNTAX>

🚨 NGUYÊN NHÂN LỖI THƯỜNG GẶP & CÁCH SỬA:
1. `ValueError: Metric not found in table`:
   - Chuỗi tìm kiếm trong `str.contains` quá dài hoặc không khớp chính xác với chỉ tiêu trong bảng.
   - HÃY NHÌN VÀO DANH SÁCH MẪU CHỈ TIÊU THỰC TẾ (bên dưới) để chọn từ khóa ngắn gọn, cốt lõi hơn.
2. `re.error: missing ), unterminated subpattern`:
   - BẮT BUỘC thêm `regex=False` vào `str.contains(..., regex=False)`.

QUY TẮC BẮT BUỘC:
1. BẮT BUỘC dùng `regex=False` trong mọi lệnh `str.contains`.
2. BẮT BUỘC kiểm tra `if not filtered_df.empty:` trước khi truy xuất giá trị. Nếu rỗng, BẮT BUỘC `raise ValueError("Metric not found in table")`. KHÔNG ĐƯỢC gán `result = 0.0`!
3. CHỈ trả về code Python thuần túy. KHÔNG giải thích, KHÔNG markdown.""",
    "user_prompt_template": """Yêu cầu người dùng: {user_query}
Mục tiêu: {muc_tieu}
Nội dung cần tìm: '{noi_dung}'

DỮ LIỆU CÓ SẴN:
{files_context}

CỘT QUAN TRỌNG:
- Cột nhãn: '{label_col}'
- Cột giá trị: '{value_col}'

BIẾN ĐƯỜNG DẪN FILE:
{paths_str}

{sample_labels_str}

MÃ NGUỒN CŨ BỊ LỖI:
```python
{previous_code}
```

THÔNG BÁO LỖI / TRACEBACK:
{error_traceback}

Hãy phân tích lỗi, nhìn vào mẫu chỉ tiêu thực tế, và viết lại toàn bộ mã Python sửa lỗi:"""
}

print("✅ Prompt templates loaded (Query Parser, Schema Mapper, Code Generator, Reflection)!")

✅ Prompts templates loaded!


## 🧠 Section 3: Agent State Definition

In [10]:
class AgentState(TypedDict, total=False):
    """Shared state dictionary passed across LangGraph nodes."""
    user_query: str
    parsed_query: Dict[str, Any]
    discovered_tables: List[Dict[str, Any]]
    table_schema: List[str]                  # Danh sách tên cột của bảng tốt nhất
    first_row_values: Dict[str, str]         # Giá trị hàng đầu tiên (khi cột có tên là số)
    column_mapping: Dict[str, str]           # Ánh xạ tên cột (label_column, value_column)
    schema: Dict[str, Any]                   # Phân tích schema bảng: useful_columns + sub_sections
    generated_code: str
    execution_result: Any
    error_traceback: Optional[str]
    retry_count: int
    status: Literal["pending", "success", "error"]
    error_message: Optional[str]
    node_latencies: Dict[str, float]

print("✅ AgentState definition updated with schema field!")

✅ AgentState TypedDict defined!


## 🧩 Section 4: Agent Pipeline Nodes

### Node 1: Query Parser Node
Trích xuất `ten_cong_ty`, `so_nam`, `noi_dung`, `thao_tac`, `tieu_chi_phu` từ câu hỏi người dùng.

In [11]:
import re
import time
import json
from typing import Dict, Any, Optional
from langchain_core.messages import SystemMessage, HumanMessage

def _normalize_company_name(company_input: str, user_query: str) -> str:
    """Normalize company name or raw ticker input against rag_module/code_stock.csv map."""
    company_input = company_input.strip() if company_input else ""
    user_query = user_query.strip() if user_query else ""
    q_lower = user_query.lower()

    try:
        from pathlib import Path
        import pandas as pd

        possible_paths = [
            Path("rag_module/code_stock.csv"),
            Path("rag_module/ViFinQA/code_stock.csv"),
            Path("/kaggle/working/r2AI_2026/rag_module/code_stock.csv"),
            Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data/code_stock.csv"),
        ]

        csv_path = None
        for p in possible_paths:
            if p.exists():
                csv_path = p
                break

        name_to_code = []
        all_tickers = set()

        if csv_path:
            df = pd.read_csv(csv_path, encoding="utf-8-sig", dtype=str)
            ticker_col = next((c for c in df.columns if "CK" in c.upper()), df.columns[0])
            name_col = next((c for c in df.columns if "TÊN" in c.upper() or "TEN" in c.upper()), df.columns[1])

            for _, row in df.iterrows():
                code = str(row[ticker_col]).strip().upper() if pd.notna(row[ticker_col]) else ""
                name = str(row[name_col]).strip() if pd.notna(row[name_col]) else ""
                if code:
                    all_tickers.add(code)
                if code and name:
                    name_to_code.append((name, code))
                    clean_name = re.sub(r"\b(CTCP|Tập đoàn|Công ty|Cổ phần|Ngân hàng|TMCP|\-\s*CTCP)\b", "", name, flags=re.IGNORECASE).strip(" -")
                    if clean_name and clean_name.lower() != name.lower() and len(clean_name) >= 3:
                        name_to_code.append((clean_name, code))
        else:
            from rag_module.search_engine import _ensure_resources, _company_map
            _ensure_resources()
            if _company_map:
                name_to_code = list(_company_map)
                all_tickers = {code.upper() for _, code in _company_map}

        # 1. Tra cứu các mã chứng khoán 3-5 ký tự in hoa xuất hiện trực tiếp trong câu hỏi người dùng
        for w in re.findall(r"\b[A-Za-z]{3,5}\b", user_query):
            if w.upper() in all_tickers:
                return w.upper()

        # 2. Sắp xếp danh sách tên công ty theo độ dài giảm dần (longest-match first) và khớp vào user_query
        name_to_code.sort(key=lambda x: len(x[0]), reverse=True)
        for name, code in name_to_code:
            if len(name) >= 3 and name.lower() in q_lower:
                return code

        # 3. Nếu company_input xuất hiện trong user_query, kiểm tra tên/mã
        if company_input:
            c_upper = company_input.upper()
            if c_upper in all_tickers and c_upper.lower() in q_lower:
                return c_upper
            c_lower = company_input.lower()
            for name, code in name_to_code:
                if len(name) >= 3 and (name.lower() in c_lower or c_lower in name.lower()):
                    if name.lower() in q_lower or code.lower() in q_lower:
                        return code

    except Exception as e:
        print(f"⚠️ [Query Parser] Stock code normalization error: {e}")

    if company_input and company_input.lower() not in q_lower and company_input.upper() not in user_query:
        return ""

    return company_input

def _clean_financial_content(text: str) -> str:
    """Clean action phrases, measurement prefixes, and query noise from financial content string."""
    if not text:
        return ""
    cleaned = text.strip()
    strip_patterns = [
        r"^tốc\s+độ\s+tăng\s+trưởng\s*%\s*",
        r"^tốc\s+độ\s+tăng\s+trưởng\s*",
        r"^tăng\s+trưởng\s*%\s*",
        r"^tăng\s+trưởng\s*",
        r"^tỷ\s+lệ\s+tăng\s+trưởng\s*",
        r"^mức\s+biến\s+động\s*",
        r"^chênh\s+lệch\s*",
        r"^so\s+sánh\s*",
        r"^tính\s+tổng\s*",
        r"^trích\s+xuất\s*",
        r"^cho\s+biết\s*",
    ]
    for pattern in strip_patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)
    trailing_patterns = [
        r"\s*là\s+bao\s+nhiêu\??$",
        r"\s*bao\s+nhiêu\??$",
        r"\s*thay\s+đổi\s+như\s+thế\s+nào\??$",
        r"\s*như\s+thế\s+nào\??$",
    ]
    for pattern in trailing_patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)
    return cleaned.strip()

def _fallback_parse_query(user_query: str) -> Dict[str, Any]:
    """Fallback rule-based parser when LLM is unreachable."""
    q = user_query.strip()
    range_match = re.search(r"từ\s*(?:năm\s*)?(\d{4})\s*đến\s*(?:năm\s*)?(\d{4})", q, re.IGNORECASE)
    if range_match:
        y1, y2 = int(range_match.group(1)), int(range_match.group(2))
        start_y, end_y = min(y1, y2), max(y1, y2)
        years = [str(y) for y in range(start_y, end_y + 1)]
        tieu_chi_phu = range_match.group(0)
    else:
        years = re.findall(r"\b(20\d{2})\b", q)
        tieu_chi_phu = None

    q_lower = q.lower()
    if "so sánh" in q_lower or "thay đổi" in q_lower or "tăng trưởng" in q_lower or "từ năm" in q_lower or "đến năm" in q_lower:
        thao_tac = "so_sanh"
    else:
        thao_tac = "trich_xuat"

    m_ticker = re.search(r"\b([A-Z]{3,5})\b", q)
    company = m_ticker.group(1) if m_ticker else ""
    company = _normalize_company_name(company, user_query)

    clean_content = re.sub(r"\b(20\d{2})\b", "", q)
    if company:
        clean_content = re.sub(rf"\b{company}\b", "", clean_content, flags=re.IGNORECASE)
    stop_phrases = [
        "tốc độ tăng trưởng %", "tốc độ tăng trưởng", "tăng trưởng %", "tăng trưởng",
        "so sánh", "từ năm", "đến năm", "của", "năm", "báo cáo", "tài chính",
        "cho", "là bao nhiêu", "bao nhiêu"
    ]
    for word in stop_phrases:
        clean_content = re.sub(rf"\b{re.escape(word)}\b", "", clean_content, flags=re.IGNORECASE)
    clean_content = _clean_financial_content(clean_content or q)

    return {
        "ticker": company,
        "ten_cong_ty": company,
        "year": ", ".join(years),
        "so_nam": years,
        "metric": clean_content or q,
        "noi_dung": clean_content or q,
        "thao_tac": thao_tac,
        "muc_tieu": thao_tac,
        "tieu_chi_phu": tieu_chi_phu,
    }

def parse_query_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    user_query = state.get("user_query", "").strip()

    if not user_query:
        return {
            **state,
            "status": "error",
            "error_message": "User query is empty.",
            "parsed_query": {},
        }

    try:
        prompt_data = PROMPT_QUERY_PARSER
        sys_prompt = prompt_data.get("system_prompt", "")
        few_shot = json.dumps(prompt_data.get("few_shot_examples", []), ensure_ascii=False, indent=2)
        
        user_content = prompt_data.get("user_prompt_template", "{user_query}").format(
            user_query=user_query,
            few_shot_examples=few_shot
        )

        llm = get_llm(cfg, temperature=0.1, timeout=30)
        response = llm.invoke([
            SystemMessage(content=sys_prompt),
            HumanMessage(content=user_content)
        ])
        raw_output = response.content if hasattr(response, "content") else str(response)
        parsed = safe_parse_json(raw_output)

        ticker_val = parsed.get("ticker") or parsed.get("ten_cong_ty") or ""
        metric_val = parsed.get("metric") or parsed.get("noi_dung") or ""
        year_val = parsed.get("year") if "year" in parsed else parsed.get("so_nam")

        if not year_val or year_val is None or str(year_val).strip() in ["None", "null", ""]:
            year_val = re.findall(r"\b(20\d{2})\b", user_query)

        if isinstance(year_val, str):
            so_nam_list = [y.strip() for y in year_val.replace(",", " ").split() if y.strip().isdigit()]
            if not so_nam_list:
                so_nam_list = re.findall(r"\b(20\d{2})\b", user_query)
        elif isinstance(year_val, (int, float)):
            so_nam_list = [str(int(year_val))]
        elif isinstance(year_val, list):
            so_nam_list = [str(y).strip() for y in year_val if str(y).strip().isdigit()]
        else:
            so_nam_list = re.findall(r"\b(20\d{2})\b", user_query)

        parsed["ticker"] = ticker_val
        parsed["ten_cong_ty"] = _normalize_company_name(ticker_val, user_query)
        parsed["year"] = ", ".join(so_nam_list) if so_nam_list else ""
        parsed["so_nam"] = so_nam_list
        parsed["metric"] = metric_val
        parsed["noi_dung"] = metric_val

        thao_tac = parsed.get("thao_tac") or parsed.get("muc_tieu") or ("so_sanh" if len(so_nam_list) > 1 or "so sánh" in user_query.lower() or "tăng trưởng" in user_query.lower() else "trich_xuat")
        if thao_tac not in ["trich_xuat", "so_sanh"]:
            thao_tac = "trich_xuat"
        
        parsed["thao_tac"] = thao_tac
        parsed["muc_tieu"] = thao_tac

        raw_noi_dung = parsed.get("noi_dung", "")
        if thao_tac == "so_sanh":
            parsed["noi_dung"] = _clean_financial_content(raw_noi_dung) or raw_noi_dung
            parsed["metric"] = parsed["noi_dung"]

        if "tieu_chi_phu" not in parsed:
            parsed["tieu_chi_phu"] = None

        print(f"📍 Node: [QUERY_PARSER] (Thời gian chạy: {time.time() - start_time:.3f}s)")
        print(f"   Công ty / Ticker: '{parsed.get('ten_cong_ty')}' (gốc: '{ticker_val}')")
        print(f"   Số năm / Year: {parsed.get('so_nam')}")
        print(f"   Nội dung / Metric: '{parsed.get('noi_dung')}'")
        print(f"   Thao tác: {parsed.get('thao_tac')}")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)

        return {
            **state,
            "parsed_query": parsed,
            "status": "pending",
            "node_latencies": node_latencies,
        }
    except Exception as exc:
        print(f"⚠️ [Query Parser] LLM không phản hồi ({exc}). Sử dụng Fallback Parser...")
        parsed = _fallback_parse_query(user_query)
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)
        return {
            **state,
            "parsed_query": parsed,
            "status": "pending",
            "node_latencies": node_latencies,
        }

query_parser_node = parse_query_node


### Node 2: Data Discovery Node
Dùng `Search Engine` tra cứu bảng dữ liệu tương ứng theo công ty, năm, nội dung.

In [12]:
# Các cột metadata/thông tin chung - dùng để lọc khi trích xuất first_row_values
_METADATA_COLUMNS = {
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
}


def _extract_table_schema(csv_path: str) -> Dict[str, Any]:
    """Đọc schema (tên cột) và giá trị hàng đầu tiên nếu có cột tên là số.

    Returns:
        Dict với 2 key:
        - table_schema: list[str] — danh sách tên cột
        - first_row_values: dict[str, str] — giá trị hàng đầu tiên (chỉ khi có cột tên số)
    """
    result: Dict[str, Any] = {"table_schema": [], "first_row_values": {}}
    try:
        df = pd.read_csv(csv_path, nrows=1)
        columns = list(df.columns)
        result["table_schema"] = columns

        # Kiểm tra xem có cột nào tên là số (0, 1, 2, 3...) không
        numeric_cols = [c for c in columns if str(c).strip().isdigit()]
        if numeric_cols and not df.empty:
            first_row = df.iloc[0]
            # Trả về giá trị hàng đầu tiên cho các cột không phải metadata
            result["first_row_values"] = {
                str(c): str(first_row[c])
                for c in columns
                if c not in _METADATA_COLUMNS
            }
    except Exception as e:
        print(f"⚠️ Không đọc được schema từ {csv_path}: {e}")
    return result


def _resolve_csv_path(csv_path_str: str, cfg: Config) -> Optional[Path]:
    """Resolve csv_path from search engine result to an actual local file path in Kaggle."""
    if not csv_path_str:
        return None
    p_str = csv_path_str.replace("\\", "/")
    direct = Path(p_str).resolve()
    if direct.exists():
        return direct
    idx_fin = p_str.find("ViFinQA")
    if idx_fin != -1:
        relative_part = p_str[idx_fin:]
        repo_root = Path("/kaggle/working/r2AI_2026").resolve()
        candidate1 = (repo_root / relative_part).resolve()
        if candidate1.exists():
            return candidate1
        candidate2 = (repo_root / "rag_module" / relative_part).resolve()
        if candidate2.exists():
            return candidate2
        candidate3 = Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data") / relative_part
        if candidate3.exists():
            return candidate3
    return None

def _log_candidates(results: List[Dict[str, Any]], year_label: str = "") -> None:
    prefix = f" (Năm {year_label})" if year_label else ""
    print(f"   📋 Danh sách {len(results)} bảng ứng viên Top-K từ Search Engine{prefix}:")
    for idx, item in enumerate(results, 1):
        p_str = item.get("csv_path", "")
        file_name = Path(p_str).name if p_str else "N/A"
        ten_bang = item.get("Ten_Bang", "N/A")
        rrf = item.get("rrf_score", 0.0)
        sample = str(item.get("matched_sample", "N/A"))
        if len(sample) > 40:
            sample = sample[:37] + "..."
        print(f"      #{idx} RRF: {rrf:.6f} | File: {file_name} | Hàng khớp: '{sample}' | Tên bảng: {ten_bang}")

def clean_query_content(noi_dung_input: str, ticker: str = "", so_nam: list = None) -> str:
    if not noi_dung_input:
        return ""
    text = str(noi_dung_input)
    text = re.sub(r"\([A-Za-z]{2,5}\)", "", text)
    text = re.sub(r"\b20\d{2}\b", "", text)
    if ticker:
        text = re.sub(r"\b" + re.escape(ticker) + r"\b", "", text, flags=re.IGNORECASE)

    patterns = [
        r"là bao nhiêu.*", r"bao nhiêu.*", r"của công ty mẹ.*", r"của ngân hàng.*",
        r"của ctcp.*", r"của tập đoàn.*", r"của công ty.*", r"vào ngày.*",
        r"đến ngày.*", r"tại ngày.*", r"cuối năm.*", r"đầu năm.*",
        r"trong năm.*", r"năm.*", r"báo cáo tài chính.*", r"báo cáo riêng.*", r"báo cáo hợp nhất.*",
    ]
    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    prefix_patterns = [
        r"^\s*tổng\s+số\s+", r"^\s*tổng\s+", r"^\s*số\s+dư\s+",
        r"^\s*giá\s+trị\s+", r"^\s*chỉ\s+tiêu\s+",
    ]
    for pp in prefix_patterns:
        text = re.sub(pp, "", text, flags=re.IGNORECASE)

    cleaned = text.strip(" ,.?:;\t\n")
    return cleaned if len(cleaned) >= 2 else noi_dung_input.strip()

def data_discovery_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    noi_dung_raw = parsed_query.get("noi_dung", "")
    thao_tac = parsed_query.get("thao_tac") or parsed_query.get("muc_tieu", "trich_xuat")

    if not so_nam and isinstance(user_query, str):
        so_nam = re.findall(r"\b(20\d{2})\b", user_query)
    if isinstance(user_query, str) and not ten_cong_ty:
        m_ticker = re.search(r"\b([A-Za-z]{3,5})\b", user_query)
        if m_ticker:
            ten_cong_ty = m_ticker.group(1).upper()

    if not noi_dung_raw:
        noi_dung_raw = user_query if isinstance(user_query, str) else ""

    noi_dung = clean_query_content(noi_dung_raw, ten_cong_ty, so_nam)

    print(f"\n🔍 [Data Discovery] Bắt đầu tìm kiếm dữ liệu...")
    print(f"   - Công ty: '{ten_cong_ty}', Số năm: {so_nam}, Nội dung đã làm sạch: '{noi_dung}' (gốc: '{noi_dung_raw}')")

    report_type = None
    if isinstance(user_query, str):
        q_lower = user_query.lower()
        if "hợp nhất" in q_lower and "riêng" not in q_lower:
            report_type = "consolidated"
        elif "báo cáo riêng" in q_lower:
            report_type = "separate"

    all_discovered_tables: List[Dict[str, Any]] = []

    try:
        from rag_module.search_engine import search_by_company_and_content
        import rag_module.search_engine as se
        se._ensure_resources()

        if not so_nam:
            results = search_by_company_and_content(
                company_name=ten_cong_ty, content=noi_dung, year=None, report_type=report_type, top_k=5
            )
            if not results and report_type is not None:
                results = search_by_company_and_content(
                    company_name=ten_cong_ty, content=noi_dung, year=None, report_type=None, top_k=5
                )
            if results:
                _log_candidates(results)
                for match in results[:3]:
                    csv_path = _resolve_csv_path(match.get("csv_path", ""), cfg)
                    if csv_path:
                        table_entry = {
                            "csv_path": str(csv_path),
                            "Ten_Bang": match.get("Ten_Bang", ""),
                            "rrf_score": match.get("rrf_score", 0.0),
                            "Ma_Doanh_Nghiep": match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                            "Nam_Tai_Chinh": match.get("Nam_Tai_Chinh", ""),
                            "Loai_Bao_Cao": match.get("Loai_Bao_Cao", ""),
                            "matched_sample": match.get("matched_sample", ""),
                        }
                        if not any(t["csv_path"] == str(csv_path) for t in all_discovered_tables):
                            all_discovered_tables.append(table_entry)
        else:
            for year in so_nam:
                results = search_by_company_and_content(
                    company_name=ten_cong_ty, content=noi_dung, year=str(year), report_type=report_type, top_k=5
                )
                if not results and report_type is not None:
                    results = search_by_company_and_content(
                        company_name=ten_cong_ty, content=noi_dung, year=str(year), report_type=None, top_k=5
                    )
                if results:
                    _log_candidates(results, year_label=str(year))
                    for match in results[:3]:
                        csv_path = _resolve_csv_path(match.get("csv_path", ""), cfg)
                        if csv_path:
                            table_entry = {
                                "csv_path": str(csv_path),
                                "Ten_Bang": match.get("Ten_Bang", ""),
                                "rrf_score": match.get("rrf_score", 0.0),
                                "Ma_Doanh_Nghiep": match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                                "Nam_Tai_Chinh": str(year),
                                "Loai_Bao_Cao": match.get("Loai_Bao_Cao", ""),
                                "matched_sample": match.get("matched_sample", ""),
                            }
                            if not any(t["csv_path"] == str(csv_path) for t in all_discovered_tables):
                                all_discovered_tables.append(table_entry)
    except Exception as e:
        print(f"⚠️ [Data Discovery] Lỗi Search Engine: {e}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["data_discovery"] = round(latency, 3)

    if not all_discovered_tables:
        return {
            **state,
            "status": "error",
            "error_message": "Không tìm thấy bảng dữ liệu phù hợp.",
            "discovered_tables": [],
            "matched_table_path": None,
            "table_schema": [],
            "first_row_values": {},
            "node_latencies": node_latencies,
        }

    first_table_path = all_discovered_tables[0]["csv_path"]

    # Extract schema và giá trị hàng đầu tiên từ bảng tốt nhất
    schema_info = _extract_table_schema(first_table_path)
    all_discovered_tables[0]["table_schema"] = schema_info["table_schema"]
    all_discovered_tables[0]["first_row_values"] = schema_info["first_row_values"]

    print(f"\n📊 [Kết quả - Data Discovery]: Đã chọn {len(all_discovered_tables)} bảng có độ khớp cao nhất.")
    print(f"   📋 Schema bảng: {schema_info['table_schema']}")
    if schema_info["first_row_values"]:
        print(f"   📋 Giá trị hàng đầu tiên (cột số): {schema_info['first_row_values']}")
    print()

    return {
        **state,
        "discovered_tables": all_discovered_tables,
        "matched_table_path": first_table_path,
        "table_schema": schema_info["table_schema"],
        "first_row_values": schema_info["first_row_values"],
        "status": "pending",
        "node_latencies": node_latencies,
    }

### Node 3: Schema Mapper Node
Ánh xạ tiêu chí phụ (tieu_chi_phu) sang tên cột thực tế trong bảng dữ liệu.
Phân tích schema bảng: useful_columns (các cột giá trị) + sub_sections (danh mục con) và sinh mô tả cột.

In [ ]:
# ==============================================================================
# Node 3: Schema Mapper Node
# ==============================================================================
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Optional, Set
from thefuzz import process, fuzz
from langchain_core.messages import SystemMessage, HumanMessage

DEFAULT_VALUE_COLUMNS = [
    "Năm nay", "Năm trước",
    "Số cuối năm", "Số đầu năm",
    "Số cuối kỳ", "Số đầu kỳ",
    "Kỳ này", "Kỳ trước",
]

METADATA_HEADER_COLUMNS = [
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
]

KNOWN_LABEL_COLUMNS = [
    "CHÍ TIÊU", "CHỈ TIÊU", "TÀI SẢN", "NGUỒN VỐN",
    "Cột_0", "Chỉ tiêu", "Mã số", "STT"
]

def _get_columns_from_table(table: Dict[str, Any]) -> List[str]:
    csv_path = table.get("csv_path", "")
    if not csv_path:
        return []
    try:
        df = pd.read_csv(csv_path, nrows=2)
        return list(df.columns)
    except Exception:
        return []

def _find_label_column(columns: List[str]) -> Optional[str]:
    for c in KNOWN_LABEL_COLUMNS:
        if c in columns:
            return c
    for c in columns:
        if c not in METADATA_HEADER_COLUMNS:
            return c
    return columns[0] if columns else None

def _find_value_column(columns: List[str], label_col: Optional[str] = None, tieu_chi_phu: Optional[str] = None) -> Optional[str]:
    label_idx = columns.index(label_col) if label_col and label_col in columns else -1
    value_candidate_cols = []
    for idx, c in enumerate(columns):
        if c in METADATA_HEADER_COLUMNS or c == label_col:
            continue
        if idx > label_idx or label_idx == -1:
            value_candidate_cols.append(c)

    if tieu_chi_phu and value_candidate_cols:
        clean_tcp = str(tieu_chi_phu).strip().lower()
        for col in value_candidate_cols:
            if clean_tcp in col.strip().lower():
                return col
        match, score = process.extractOne(tieu_chi_phu, value_candidate_cols, scorer=fuzz.token_set_ratio)
        if score >= 50:
            return match

    for c in DEFAULT_VALUE_COLUMNS:
        if c in value_candidate_cols:
            return c
    if value_candidate_cols:
        return value_candidate_cols[0]
    return None

def _extract_useful_columns(df: pd.DataFrame, label_col: Optional[str], metadata_cols: Set[str]) -> List[Dict[str, str]]:
    useful = []
    all_columns = list(df.columns)
    for col in all_columns:
        if col in metadata_cols or col == label_col:
            continue
        numeric_vals = pd.to_numeric(df[col], errors="coerce")
        non_null_count = df[col].notna().sum()
        if non_null_count == 0:
            continue
        numeric_ratio = numeric_vals.notna().sum() / non_null_count
        if numeric_ratio < 0.5:
            continue
        col_name = str(col)
        if col_name.strip().isdigit():
            resolved_parts = []
            for row_idx in range(min(5, len(df))):
                cell_val = df.iloc[row_idx][col]
                if pd.notna(cell_val):
                    cell_str = str(cell_val).strip()
                    if not cell_str or cell_str.lower() in ["nan", "none", "null", "n/a", "-", "—"]:
                        continue
                    try:
                        float(cell_str.replace(",", "").replace(".", ""))
                        break
                    except ValueError:
                        resolved_parts.append(cell_str)
                else:
                    continue
            if resolved_parts:
                col_name = " - ".join(resolved_parts)
                print(f"      🔹 Cột số '{col}' → Giải mã tên cột thật: '{col_name}'")
        useful.append({"column_name": col_name, "column_description": ""})
    return useful

def _extract_sub_sections(df: pd.DataFrame, label_col: Optional[str], metadata_cols: Set[str]) -> List[Dict[str, Any]]:
    if not label_col or label_col not in df.columns:
        return []
    value_cols = [c for c in df.columns if c not in metadata_cols and c != label_col]
    if not value_cols:
        return []
    sections = []
    section_header_rows = []
    prev_was_empty = False
    for idx in range(len(df)):
        label_val = df.iloc[idx][label_col]
        is_label_empty = pd.isna(label_val) or str(label_val).strip() == ""
        if not is_label_empty and prev_was_empty:
            label_text = str(label_val).strip()
            total_value = None
            for vc in value_cols:
                cell = df.iloc[idx][vc]
                if pd.notna(cell):
                    try:
                        cell_str = str(cell).strip().replace(",", "")
                        if cell_str.startswith("(") and cell_str.endswith(")"):
                            cell_str = "-" + cell_str[1:-1].strip()
                        total_value = float(cell_str)
                        break
                    except (ValueError, TypeError):
                        continue
            section_header_rows.append({"row_idx": idx, "section_name": label_text, "total_value": total_value})
        prev_was_empty = is_label_empty

    for i, header in enumerate(section_header_rows):
        start = header["row_idx"] + 1
        end = section_header_rows[i + 1]["row_idx"] - 1 if i + 1 < len(section_header_rows) else len(df) - 1
        while end >= start:
            val = df.iloc[end][label_col]
            if pd.isna(val) or str(val).strip() == "":
                end -= 1
            else:
                break
        if start <= end:
            sections.append({"section_name": header["section_name"], "range": [start, end], "total_value": header["total_value"]})
    return sections

def _enrich_column_descriptions(cfg: Config, useful_columns: List[Dict[str, str]], table_name: str) -> List[Dict[str, str]]:
    if not useful_columns:
        return useful_columns
    try:
        prompt_data = PROMPT_SCHEMA_MAPPER
        system_prompt = prompt_data.get("system_prompt", "")
        user_template = prompt_data.get("user_prompt_template", "")
        col_names = [c["column_name"] for c in useful_columns]
        user_content = user_template.format(table_name=table_name, columns=", ".join(col_names))
        print(f"   🤖 [Schema Mapper] Đang gọi LLM để sinh mô tả ngữ nghĩa cho {len(col_names)} cột...")
        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_content)])
        raw_text = response.content if isinstance(response.content, str) else str(response.content)
        parsed = safe_parse_json(raw_text)
        descriptions_list = parsed if isinstance(parsed, list) else parsed.get("columns", [])
        desc_map = {}
        for item in descriptions_list:
            if isinstance(item, dict):
                desc_map[item.get("column_name", "")] = item.get("column_description", "")
        for col in useful_columns:
            if col["column_name"] in desc_map:
                col["column_description"] = desc_map[col["column_name"]]
        print(f"   ✅ [Schema Mapper] LLM đã làm giàu mô tả cho {len(desc_map)}/{len(useful_columns)} cột thành công.")
    except Exception as e:
        print(f"   ⚠️ [Schema Mapper] LLM enrichment failed: {e}. Giữ descriptions rỗng.")
    return useful_columns

def schema_mapper_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 3: Map tiêu chí phụ sang tên cột thực tế + phân tích schema bảng."""
    cfg = cfg or config
    start_time = time.time()
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    print(f"\n" + "=" * 65)
    print(f"🔍 [Node 3: SCHEMA MAPPER] Bắt đầu phân tích Schema...")
    print(f"=" * 65)
    print(f"   - Tiêu chí phụ cần tìm: '{tieu_chi_phu or '(không có)'}'")

    column_mapping: Dict[str, str] = {}
    schema: Dict[str, Any] = {"useful_columns": [], "sub_sections": []}

    if not discovered_tables:
        print(f"   ⚠️ [Schema Mapper] Không có bảng dữ liệu đầu vào để ánh xạ.")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "schema": schema, "status": "pending", "node_latencies": node_latencies}

    first_table = discovered_tables[0]
    columns = _get_columns_from_table(first_table)
    if not columns:
        print(f"   ⚠️ [Schema Mapper] Không đọc được cột từ file: {first_table.get('csv_path')}")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "schema": schema, "status": "pending", "node_latencies": node_latencies}

    print(f"   📋 Cột gốc trong bảng ({len(columns)} cột): {columns}")

    label_col = _find_label_column(columns)
    if label_col:
        column_mapping["label_column"] = label_col
        print(f"   🏷️ Cột nhãn (chỉ tiêu tài chính): '{label_col}'")

    value_col = _find_value_column(columns, label_col, tieu_chi_phu)
    if value_col:
        column_mapping["value_column"] = value_col
        print(f"   🎯 Cột giá trị ưu tiên: '{value_col}'")

    column_mapping["all_columns"] = str(columns)

    # Schema Analysis
    csv_path = first_table.get("csv_path", "")
    table_name = first_table.get("Ten_Bang", Path(csv_path).stem if csv_path else "unknown")
    metadata_set = set(METADATA_HEADER_COLUMNS)
    try:
        df_full = pd.read_csv(csv_path)
        print(f"\n   📐 Phân tích chi tiết bảng '{table_name}' ({len(df_full)} dòng, {len(df_full.columns)} cột):")
        useful_columns = _extract_useful_columns(df_full, label_col, metadata_set)
        print(f"   📊 Cột giá trị hữu dụng phát hiện được ({len(useful_columns)} cột): {[c['column_name'] for c in useful_columns]}")
        sub_sections = _extract_sub_sections(df_full, label_col, metadata_set)
        print(f"   📂 Danh mục con (Sub-sections) tìm thấy ({len(sub_sections)} mục):")
        for sec in sub_sections[:6]:
            total_str = f"{sec['total_value']:,.2f}" if isinstance(sec['total_value'], (int, float)) else "N/A"
            print(f"      • {sec['section_name']} -> Hàng {sec['range'][0]}..{sec['range'][1]} | Total: {total_str}")
        if len(sub_sections) > 6:
            print(f"      ... và {len(sub_sections) - 6} danh mục con khác")
        useful_columns = _enrich_column_descriptions(cfg, useful_columns, table_name)
        schema = {"useful_columns": useful_columns, "sub_sections": sub_sections}
        print(f"\n   📋 [Kết quả Schema Phân tích]:")
        for col in useful_columns:
            desc = col['column_description'] or '(không có mô tả)'
            print(f"      • Cột '{col['column_name']}': {desc}")
    except Exception as e:
        print(f"   ⚠️ [Schema Mapper] Lỗi trong quá trình phân tích schema: {e}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["schema_mapper"] = round(latency, 3)
    print(f"\n   ⏱️ [Schema Mapper] Hoàn thành trong {latency:.3f}s")
    print(f"=" * 65 + "\n")
    return {**state, "column_mapping": column_mapping, "schema": schema, "status": "pending", "node_latencies": node_latencies}

print("✅ Node 3: Schema Mapper loaded with full logging!")

### Node 3: Schema Mapper Node
Ánh xạ tiêu chí phụ (tieu_chi_phu) sang tên cột thực tế trong bảng dữ liệu.
Phân tích schema bảng: useful_columns (các cột giá trị) + sub_sections (danh mục con) và sinh mô tả cột.

In [ ]:
# ==============================================================================
# Node 3: Schema Mapper Node
# ==============================================================================
import time
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional, Set
from thefuzz import process, fuzz
from langchain_core.messages import SystemMessage, HumanMessage

DEFAULT_VALUE_COLUMNS = [
    "Năm nay", "Năm trước",
    "Số cuối năm", "Số đầu năm",
    "Số cuối kỳ", "Số đầu kỳ",
    "Kỳ này", "Kỳ trước",
]

METADATA_HEADER_COLUMNS = [
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
]

KNOWN_LABEL_COLUMNS = [
    "CHÍ TIÊU", "CHỈ TIÊU", "TÀI SẢN", "NGUỒN VỐN",
    "Cột_0", "Chỉ tiêu", "Mã số", "STT"
]

def _get_columns_from_table(table: Dict[str, Any]) -> List[str]:
    csv_path = table.get("csv_path", "")
    if not csv_path:
        return []
    try:
        df = pd.read_csv(csv_path, nrows=2)
        return list(df.columns)
    except Exception:
        return []

def _find_label_column(columns: List[str]) -> Optional[str]:
    for c in KNOWN_LABEL_COLUMNS:
        if c in columns:
            return c
    for c in columns:
        if c not in METADATA_HEADER_COLUMNS:
            return c
    return columns[0] if columns else None

def _find_value_column(columns: List[str], label_col: Optional[str] = None, tieu_chi_phu: Optional[str] = None) -> Optional[str]:
    label_idx = columns.index(label_col) if label_col and label_col in columns else -1
    value_candidate_cols = []
    for idx, c in enumerate(columns):
        if c in METADATA_HEADER_COLUMNS or c == label_col:
            continue
        if idx > label_idx or label_idx == -1:
            value_candidate_cols.append(c)

    if tieu_chi_phu and value_candidate_cols:
        clean_tcp = str(tieu_chi_phu).strip().lower()
        for col in value_candidate_cols:
            if clean_tcp in col.strip().lower():
                return col
        match, score = process.extractOne(tieu_chi_phu, value_candidate_cols, scorer=fuzz.token_set_ratio)
        if score >= 50:
            return match

    for c in DEFAULT_VALUE_COLUMNS:
        if c in value_candidate_cols:
            return c
    if value_candidate_cols:
        return value_candidate_cols[0]
    return None

def _extract_useful_columns(df: pd.DataFrame, label_col: Optional[str], metadata_cols: Set[str]) -> List[Dict[str, str]]:
    useful = []
    all_columns = list(df.columns)
    for col in all_columns:
        if col in metadata_cols or col == label_col:
            continue
        numeric_vals = pd.to_numeric(df[col], errors="coerce")
        non_null_count = df[col].notna().sum()
        if non_null_count == 0:
            continue
        numeric_ratio = numeric_vals.notna().sum() / non_null_count
        if numeric_ratio < 0.5:
            continue
        col_name = str(col)
        if col_name.strip().isdigit():
            resolved_parts = []
            for row_idx in range(min(5, len(df))):
                cell_val = df.iloc[row_idx][col]
                if pd.notna(cell_val):
                    cell_str = str(cell_val).strip()
                    if not cell_str or cell_str.lower() in ["nan", "none", "null", "n/a", "-", "—"]:
                        continue
                    try:
                        float(cell_str.replace(",", "").replace(".", ""))
                        break
                    except ValueError:
                        resolved_parts.append(cell_str)
                else:
                    continue
            if resolved_parts:
                col_name = " - ".join(resolved_parts)
        useful.append({"column_name": col_name, "column_description": ""})
    return useful

def _extract_sub_sections(df: pd.DataFrame, label_col: Optional[str], metadata_cols: Set[str]) -> List[Dict[str, Any]]:
    if not label_col or label_col not in df.columns:
        return []
    value_cols = [c for c in df.columns if c not in metadata_cols and c != label_col]
    if not value_cols:
        return []
    sections = []
    section_header_rows = []
    prev_was_empty = True
    for idx in range(len(df)):
        label_val = df.iloc[idx][label_col]
        is_label_empty = pd.isna(label_val) or str(label_val).strip() == ""
        if not is_label_empty and prev_was_empty:
            label_text = str(label_val).strip()
            total_value = None
            for vc in value_cols:
                cell = df.iloc[idx][vc]
                if pd.notna(cell):
                    try:
                        cell_str = str(cell).strip().replace(",", "")
                        if cell_str.startswith("(") and cell_str.endswith(")"):
                            cell_str = "-" + cell_str[1:-1].strip()
                        total_value = float(cell_str)
                        break
                    except (ValueError, TypeError):
                        continue
            section_header_rows.append({"row_idx": idx, "section_name": label_text, "total_value": total_value})
        prev_was_empty = is_label_empty

    for i, header in enumerate(section_header_rows):
        start = header["row_idx"] + 1
        end = section_header_rows[i + 1]["row_idx"] - 1 if i + 1 < len(section_header_rows) else len(df) - 1
        if start <= end:
            sections.append({"section_name": header["section_name"], "range": [start, end], "total_value": header["total_value"]})
    return sections

def _enrich_column_descriptions(cfg: Config, useful_columns: List[Dict[str, str]], table_name: str) -> List[Dict[str, str]]:
    if not useful_columns:
        return useful_columns
    try:
        prompt_data = PROMPT_SCHEMA_MAPPER
        system_prompt = prompt_data.get("system_prompt", "")
        user_template = prompt_data.get("user_prompt_template", "")
        col_names = [c["column_name"] for c in useful_columns]
        user_content = user_template.format(table_name=table_name, columns=", ".join(col_names))
        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_content)])
        raw_text = response.content if isinstance(response.content, str) else str(response.content)
        parsed = safe_parse_json(raw_text)
        descriptions_list = parsed if isinstance(parsed, list) else parsed.get("columns", [])
        desc_map = {}
        for item in descriptions_list:
            if isinstance(item, dict):
                desc_map[item.get("column_name", "")] = item.get("column_description", "")
        for col in useful_columns:
            if col["column_name"] in desc_map:
                col["column_description"] = desc_map[col["column_name"]]
        print(f"   ✅ [Schema Mapper] LLM đã sinh mô tả cho {len(desc_map)} cột.")
    except Exception as e:
        print(f"   ⚠️ [Schema Mapper] LLM enrichment failed: {e}. Giữ descriptions rỗng.")
    return useful_columns

def schema_mapper_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 3: Map tiêu chí phụ sang tên cột thực tế + phân tích schema bảng."""
    cfg = cfg or config
    start_time = time.time()
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    print(f"\n🔍 [Schema Mapper] Đang ánh xạ tiêu chí → cột thực tế...")
    print(f"   - Tiêu chí phụ: {tieu_chi_phu or '(không có)'}")

    column_mapping: Dict[str, str] = {}
    schema: Dict[str, Any] = {"useful_columns": [], "sub_sections": []}

    if not discovered_tables:
        print(f"⚠️ [Schema Mapper] Không có bảng dữ liệu để ánh xạ.")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "schema": schema, "status": "pending", "node_latencies": node_latencies}

    first_table = discovered_tables[0]
    columns = _get_columns_from_table(first_table)
    if not columns:
        print(f"⚠️ [Schema Mapper] Không đọc được cột từ bảng: {first_table.get('csv_path')}")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "schema": schema, "status": "pending", "node_latencies": node_latencies}

    label_col = _find_label_column(columns)
    if label_col:
        column_mapping["label_column"] = label_col
        print(f"   - Cột nhãn (chỉ tiêu): '{label_col}'")

    value_col = _find_value_column(columns, label_col, tieu_chi_phu)
    if value_col:
        column_mapping["value_column"] = value_col
        print(f"   - Cột giá trị: '{value_col}'")

    column_mapping["all_columns"] = str(columns)

    # Schema Analysis
    csv_path = first_table.get("csv_path", "")
    table_name = first_table.get("Ten_Bang", Path(csv_path).stem if csv_path else "unknown")
    metadata_set = set(METADATA_HEADER_COLUMNS)
    try:
        df_full = pd.read_csv(csv_path)
        print(f"\n📐 [Schema Mapper] Đang phân tích schema bảng ({len(df_full)} hàng)...")
        useful_columns = _extract_useful_columns(df_full, label_col, metadata_set)
        sub_sections = _extract_sub_sections(df_full, label_col, metadata_set)
        useful_columns = _enrich_column_descriptions(cfg, useful_columns, table_name)
        schema = {"useful_columns": useful_columns, "sub_sections": sub_sections}
    except Exception as e:
        print(f"⚠️ [Schema Mapper] Lỗi phân tích schema: {e}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["schema_mapper"] = round(latency, 3)
    return {**state, "column_mapping": column_mapping, "schema": schema, "status": "pending", "node_latencies": node_latencies}

print("✅ Node 3: Schema Mapper loaded!")

### Node 3: Schema Mapper Node
Ánh xạ tiêu chí phụ (tieu_chi_phu) sang tên cột thực tế trong bảng dữ liệu.
Phân tích schema bảng: useful_columns (các cột giá trị) + sub_sections (danh mục con) và sinh mô tả cột.

In [ ]:
# ==============================================================================
# Node 3: Schema Mapper Node
# ==============================================================================
import time
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional, Set
from thefuzz import process, fuzz
from langchain_core.messages import SystemMessage, HumanMessage

DEFAULT_VALUE_COLUMNS = [
    "Năm nay", "Năm trước",
    "Số cuối năm", "Số đầu năm",
    "Số cuối kỳ", "Số đầu kỳ",
    "Kỳ này", "Kỳ trước",
]

METADATA_HEADER_COLUMNS = [
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
]

KNOWN_LABEL_COLUMNS = [
    "CHÍ TIÊU", "CHỈ TIÊU", "TÀI SẢN", "NGUỒN VỐN",
    "Cột_0", "Chỉ tiêu", "Mã số", "STT"
]

def _get_columns_from_table(table: Dict[str, Any]) -> List[str]:
    csv_path = table.get("csv_path", "")
    if not csv_path:
        return []
    try:
        df = pd.read_csv(csv_path, nrows=2)
        return list(df.columns)
    except Exception:
        return []

def _find_label_column(columns: List[str]) -> Optional[str]:
    for c in KNOWN_LABEL_COLUMNS:
        if c in columns:
            return c
    for c in columns:
        if c not in METADATA_HEADER_COLUMNS:
            return c
    return columns[0] if columns else None

def _find_value_column(columns: List[str], label_col: Optional[str] = None, tieu_chi_phu: Optional[str] = None) -> Optional[str]:
    label_idx = columns.index(label_col) if label_col and label_col in columns else -1
    value_candidate_cols = []
    for idx, c in enumerate(columns):
        if c in METADATA_HEADER_COLUMNS or c == label_col:
            continue
        if idx > label_idx or label_idx == -1:
            value_candidate_cols.append(c)

    if tieu_chi_phu and value_candidate_cols:
        clean_tcp = str(tieu_chi_phu).strip().lower()
        for col in value_candidate_cols:
            if clean_tcp in col.strip().lower():
                return col
        match, score = process.extractOne(tieu_chi_phu, value_candidate_cols, scorer=fuzz.token_set_ratio)
        if score >= 50:
            return match

    for c in DEFAULT_VALUE_COLUMNS:
        if c in value_candidate_cols:
            return c
    if value_candidate_cols:
        return value_candidate_cols[0]
    return None

def _extract_useful_columns(df: pd.DataFrame, label_col: Optional[str], metadata_cols: Set[str]) -> List[Dict[str, str]]:
    useful = []
    all_columns = list(df.columns)
    for col in all_columns:
        if col in metadata_cols or col == label_col:
            continue
        numeric_vals = pd.to_numeric(df[col], errors="coerce")
        non_null_count = df[col].notna().sum()
        if non_null_count == 0:
            continue
        numeric_ratio = numeric_vals.notna().sum() / non_null_count
        if numeric_ratio < 0.5:
            continue
        col_name = str(col)
        if col_name.strip().isdigit():
            resolved_parts = []
            for row_idx in range(min(5, len(df))):
                cell_val = df.iloc[row_idx][col]
                if pd.notna(cell_val):
                    cell_str = str(cell_val).strip()
                    try:
                        float(cell_str.replace(",", "").replace(".", ""))
                        break
                    except ValueError:
                        resolved_parts.append(cell_str)
                else:
                    break
            if resolved_parts:
                col_name = " - ".join(resolved_parts)
        useful.append({"column_name": col_name, "column_description": ""})
    return useful

def _extract_sub_sections(df: pd.DataFrame, label_col: Optional[str], metadata_cols: Set[str]) -> List[Dict[str, Any]]:
    if not label_col or label_col not in df.columns:
        return []
    value_cols = [c for c in df.columns if c not in metadata_cols and c != label_col]
    if not value_cols:
        return []
    sections = []
    section_header_rows = []
    prev_was_empty = True
    for idx in range(len(df)):
        label_val = df.iloc[idx][label_col]
        is_label_empty = pd.isna(label_val) or str(label_val).strip() == ""
        if not is_label_empty and prev_was_empty:
            label_text = str(label_val).strip()
            total_value = None
            for vc in value_cols:
                cell = df.iloc[idx][vc]
                if pd.notna(cell):
                    try:
                        cell_str = str(cell).strip().replace(",", "")
                        if cell_str.startswith("(") and cell_str.endswith(")"):
                            cell_str = "-" + cell_str[1:-1].strip()
                        total_value = float(cell_str)
                        break
                    except (ValueError, TypeError):
                        continue
            section_header_rows.append({"row_idx": idx, "section_name": label_text, "total_value": total_value})
        prev_was_empty = is_label_empty

    for i, header in enumerate(section_header_rows):
        start = header["row_idx"] + 1
        end = section_header_rows[i + 1]["row_idx"] - 1 if i + 1 < len(section_header_rows) else len(df) - 1
        if start <= end:
            sections.append({"section_name": header["section_name"], "range": [start, end], "total_value": header["total_value"]})
    return sections

def _enrich_column_descriptions(cfg: Config, useful_columns: List[Dict[str, str]], table_name: str) -> List[Dict[str, str]]:
    if not useful_columns:
        return useful_columns
    try:
        prompt_data = PROMPT_SCHEMA_MAPPER
        system_prompt = prompt_data.get("system_prompt", "")
        user_template = prompt_data.get("user_prompt_template", "")
        col_names = [c["column_name"] for c in useful_columns]
        user_content = user_template.format(table_name=table_name, columns=", ".join(col_names))
        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_content)])
        raw_text = response.content if isinstance(response.content, str) else str(response.content)
        parsed = safe_parse_json(raw_text)
        descriptions_list = parsed if isinstance(parsed, list) else parsed.get("columns", [])
        desc_map = {}
        for item in descriptions_list:
            if isinstance(item, dict):
                desc_map[item.get("column_name", "")] = item.get("column_description", "")
        for col in useful_columns:
            if col["column_name"] in desc_map:
                col["column_description"] = desc_map[col["column_name"]]
        print(f"   ✅ [Schema Mapper] LLM đã sinh mô tả cho {len(desc_map)} cột.")
    except Exception as e:
        print(f"   ⚠️ [Schema Mapper] LLM enrichment failed: {e}. Giữ descriptions rỗng.")
    return useful_columns

def schema_mapper_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 3: Map tiêu chí phụ sang tên cột thực tế + phân tích schema bảng."""
    cfg = cfg or config
    start_time = time.time()
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    print(f"\n🔍 [Schema Mapper] Đang ánh xạ tiêu chí → cột thực tế...")
    print(f"   - Tiêu chí phụ: {tieu_chi_phu or '(không có)'}")

    column_mapping: Dict[str, str] = {}
    schema: Dict[str, Any] = {"useful_columns": [], "sub_sections": []}

    if not discovered_tables:
        print(f"⚠️ [Schema Mapper] Không có bảng dữ liệu để ánh xạ.")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "schema": schema, "status": "pending", "node_latencies": node_latencies}

    first_table = discovered_tables[0]
    columns = _get_columns_from_table(first_table)
    if not columns:
        print(f"⚠️ [Schema Mapper] Không đọc được cột từ bảng: {first_table.get('csv_path')}")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "schema": schema, "status": "pending", "node_latencies": node_latencies}

    label_col = _find_label_column(columns)
    if label_col:
        column_mapping["label_column"] = label_col
        print(f"   - Cột nhãn (chỉ tiêu): '{label_col}'")

    value_col = _find_value_column(columns, label_col, tieu_chi_phu)
    if value_col:
        column_mapping["value_column"] = value_col
        print(f"   - Cột giá trị: '{value_col}'")

    column_mapping["all_columns"] = str(columns)

    # Schema Analysis
    csv_path = first_table.get("csv_path", "")
    table_name = first_table.get("Ten_Bang", Path(csv_path).stem if csv_path else "unknown")
    metadata_set = set(METADATA_HEADER_COLUMNS)
    try:
        df_full = pd.read_csv(csv_path)
        print(f"\n📐 [Schema Mapper] Đang phân tích schema bảng ({len(df_full)} hàng)...")
        useful_columns = _extract_useful_columns(df_full, label_col, metadata_set)
        sub_sections = _extract_sub_sections(df_full, label_col, metadata_set)
        useful_columns = _enrich_column_descriptions(cfg, useful_columns, table_name)
        schema = {"useful_columns": useful_columns, "sub_sections": sub_sections}
    except Exception as e:
        print(f"⚠️ [Schema Mapper] Lỗi phân tích schema: {e}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["schema_mapper"] = round(latency, 3)
    return {**state, "column_mapping": column_mapping, "schema": schema, "status": "pending", "node_latencies": node_latencies}

print("✅ Node 3: Schema Mapper loaded!")

### Node 4: Code Generator & Reflection Node
Sinh mã Pandas xử lý câu hỏi tài chính và hỗ trợ Reflection Debugging Loop khi xảy ra lỗi.

In [13]:
import re
import time
from typing import Dict, Any, Optional, List
from pathlib import Path
import pandas as pd
from langchain_core.messages import SystemMessage, HumanMessage

def clean_python_code(raw_code: str) -> str:
    """Extract clean Python code from LLM response, stripping markdown and conversational text."""
    if not raw_code:
        return ""
    pattern = r"```(?:python)?\s*\n?(.*?)\n?```"
    matches = re.findall(pattern, raw_code, re.DOTALL)
    if matches:
        return matches[0].strip()

    cleaned = raw_code.strip()
    if cleaned.startswith("```") and cleaned.endswith("```"):
        cleaned = cleaned[3:-3].strip()

    lines = cleaned.splitlines()
    code_start_idx = 0
    for idx, line in enumerate(lines):
        l = line.strip()
        if l.startswith("import ") or l.startswith("def ") or l.startswith("file_path") or l.startswith("df =") or l.startswith("result ="):
            code_start_idx = idx
            break
    return "\n".join(lines[code_start_idx:]).strip()

_METADATA_COLUMNS = {
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
}

_KNOWN_LABEL_COLUMNS = [
    "CHÍ TIÊU", "CHỈ TIÊU", "TÀI SẢN", "NGUỒN VỐN",
    "Cột_0", "Chỉ tiêu", "Mã số", "STT"
]

def _resolve_label_column(
    table_schema: List[str],
    first_row_values: Optional[Dict[str, str]] = None,
    column_mapping: Optional[Dict[str, str]] = None,
) -> str:
    column_mapping = column_mapping or {}
    first_row_values = first_row_values or {}
    mapped_label = column_mapping.get("label_column")
    if mapped_label and mapped_label in table_schema:
        return mapped_label
    if not table_schema:
        return mapped_label or "CHỈ TIÊU"
    non_meta_cols = [c for c in table_schema if c not in _METADATA_COLUMNS]
    if not non_meta_cols:
        return mapped_label or "CHỈ TIÊU"
    for known in _KNOWN_LABEL_COLUMNS:
        for c in non_meta_cols:
            if known.lower() == str(c).strip().lower():
                return c
    if first_row_values:
        for known in ["CHỈ TIÊU", "CHÍ TIÊU", "TÀI SẢN", "NGUỒN VỐN", "Chỉ tiêu"]:
            for col, val in first_row_values.items():
                if col in non_meta_cols and known.lower() in str(val).strip().lower():
                    return str(col)
    return non_meta_cols[0]

def _resolve_value_column(
    table_schema: List[str],
    first_row_values: Dict[str, str],
    parsed_query: Dict[str, Any],
    column_mapping: Dict[str, str],
    label_col: Optional[str] = None,
    schema: Optional[Dict[str, Any]] = None,
) -> str:
    fallback = column_mapping.get("value_column", "Năm nay")
    schema = schema or {}
    if not table_schema:
        return fallback
    label_col = label_col or column_mapping.get("label_column", "")
    data_cols = [c for c in table_schema if c not in _METADATA_COLUMNS and c != label_col]
    if not data_cols:
        return fallback
    if len(data_cols) == 1:
        return data_cols[0]

    useful_columns = schema.get("useful_columns", [])
    if len(useful_columns) > 1:
        tieu_chi_phu = parsed_query.get("tieu_chi_phu", "")
        noi_dung = parsed_query.get("noi_dung", "")
        search_terms = [t for t in [tieu_chi_phu, noi_dung] if t]
        for search_term in search_terms:
            search_lower = str(search_term).strip().lower()
            for uc in useful_columns:
                col_name = uc.get("column_name", "")
                col_desc = uc.get("column_description", "")
                if (search_lower in col_name.lower() or search_lower in col_desc.lower() or
                    col_name.lower() in search_lower or col_desc.lower() in search_lower):
                    if col_name in data_cols:
                        return col_name
                    for dc in data_cols:
                        if str(dc).strip().isdigit():
                            return dc

    numeric_named_cols = [c for c in data_cols if str(c).strip().isdigit()]
    if numeric_named_cols and first_row_values:
        tieu_chi_phu = parsed_query.get("tieu_chi_phu", "")
        if tieu_chi_phu:
            tieu_chi_lower = str(tieu_chi_phu).strip().lower()
            for col in data_cols:
                val = first_row_values.get(str(col), "")
                if tieu_chi_lower in val.lower():
                    return col
        for col in data_cols:
            val = first_row_values.get(str(col), "")
            if fallback.lower() in val.lower():
                return col
    if fallback in data_cols:
        return fallback
    return data_cols[0] if data_cols else fallback

def _build_files_context(
    discovered_tables: List[Dict[str, Any]],
    column_mapping: Dict[str, str],
    table_schema: List[str] = None,
    first_row_values: Dict[str, str] = None,
    schema: Dict[str, Any] = None,
) -> str:
    if not discovered_tables:
        return "Không có bảng dữ liệu."
    lines = []
    for i, tbl in enumerate(discovered_tables):
        csv_path = tbl.get("csv_path", "")
        ten_bang = tbl.get("Ten_Bang", "N/A")
        nam = tbl.get("Nam_Tai_Chinh", "N/A")
        escaped_path = csv_path.replace('\\', '\\\\')
        lines.append(f"- File {i+1} (Năm {nam}):\n  Đường dẫn: '{escaped_path}'\n  Tên bảng: {ten_bang}\n")
    lines.append(f"\nColumn Mapping: {column_mapping}")
    if table_schema:
        lines.append(f"\nSchema bảng (tên các cột): {table_schema}")
    if first_row_values:
        lines.append(f"\nGiá trị hàng đầu tiên (giúp hiểu ý nghĩa cột số):")
        for col, val in first_row_values.items():
            lines.append(f"  Cột '{col}' → '{val}'")
    schema = schema or {}
    useful_columns = schema.get("useful_columns", [])
    sub_sections = schema.get("sub_sections", [])
    if useful_columns:
        lines.append(f"\nSCHEMA PHÂN TÍCH BẢNG - CỘT GIÁ TRỊ HỮU DỤNG:")
        for uc in useful_columns:
            desc_str = f" — {uc.get('column_description', '')}" if uc.get('column_description') else ""
            lines.append(f"  • '{uc.get('column_name', '')}'{desc_str}")
    if sub_sections:
        lines.append(f"\nSCHEMA PHÂN TÍCH BẢNG - DANH MỤC CON (SUB-SECTIONS):")
        for sec in sub_sections:
            total_val = sec.get("total_value")
            total_str = f", total_value={total_val}" if total_val is not None else ", total_value=N/A"
            lines.append(f"  • '{sec.get('section_name', '')}' (hàng {sec.get('range', [])}{total_str})")
    return "\n".join(lines)

def code_generator_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 4: Sinh code Pandas hoặc sửa code lỗi (Reflection Loop)."""
    cfg = cfg or config
    start_time = time.time()
    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    column_mapping = state.get("column_mapping", {})
    table_schema = state.get("table_schema", [])
    first_row_values = state.get("first_row_values", {})
    schema = state.get("schema", {})
    error_traceback = state.get("error_traceback")
    retry_count = state.get("retry_count", 0)

    muc_tieu = parsed_query.get("muc_tieu", "trich_xuat")
    noi_dung = parsed_query.get("noi_dung", "")
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    label_col = _resolve_label_column(table_schema, first_row_values, column_mapping)
    value_col = _resolve_value_column(table_schema, first_row_values, parsed_query, column_mapping, label_col, schema=schema)

    print(f"   📋 [Code Generator] Schema: {table_schema}")
    print(f"   📋 [Code Generator] Label col: '{label_col}', Value col: '{value_col}'")
    if first_row_values:
        print(f"   📋 [Code Generator] First row values: {first_row_values}")

    files_context = _build_files_context(discovered_tables, column_mapping, table_schema, first_row_values, schema=schema)

    paths_str = ""
    if discovered_tables:
        if len(discovered_tables) == 1:
            escaped = discovered_tables[0]["csv_path"].replace('\\', '\\\\')
            paths_str = f"file_path = '{escaped}'"
        else:
            for tbl in discovered_tables:
                nam = tbl.get("Nam_Tai_Chinh", "default")
                escaped = tbl["csv_path"].replace('\\', '\\\\')
                paths_str += f"file_path_{nam} = '{escaped}'\n"

    try:
        if not error_traceback or retry_count == 0:
            prompt_data = PROMPT_CODE_GENERATOR
            system_prompt = prompt_data["system_prompt"]
            few_shots = prompt_data.get("few_shot_examples", [])
            goal_descs = prompt_data.get("goal_descriptions", {})
            goal_instructions = prompt_data.get("goal_instructions", {})

            messages = [SystemMessage(content=system_prompt)]
            for ex in few_shots:
                messages.append(HumanMessage(content=f"Yêu cầu: {ex['user_query']}\nFile Path: {ex['file_path']}\nColumn Mapping: {ex['column_mapping']}"))
                messages.append(SystemMessage(content=ex["generated_code"]))

            goal_desc = goal_descs.get(muc_tieu, muc_tieu)
            goal_inst_template = goal_instructions.get(muc_tieu, "")
            goal_inst = goal_inst_template.format(noi_dung=noi_dung, label_col=label_col, value_col=value_col) if goal_inst_template else ""

            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu_desc=goal_desc,
                noi_dung=noi_dung,
                ten_cong_ty=ten_cong_ty,
                so_nam=so_nam,
                tieu_chi_phu=tieu_chi_phu or "(không có)",
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                goal_instruction=goal_inst,
            )
            messages.append(HumanMessage(content=human_content))
        else:
            prompt_data = PROMPT_REFLECTION
            system_prompt = prompt_data["system_prompt"]
            print(f"🔄 [Reflection Loop] Đang sửa lỗi mã nguồn (Lần {retry_count})...")
            retry_forcing_msg = f"Execution failed with error: {error_traceback.strip()}\nCRITICAL: The metric was not found. Inspect sample row labels below and use a shorter core keyword."
            sample_labels = []
            if discovered_tables:
                for tbl in discovered_tables:
                    c_path = tbl.get("csv_path")
                    if c_path and Path(c_path).exists():
                        try:
                            sub_df = pd.read_csv(c_path)
                            if label_col in sub_df.columns:
                                labels = sub_df[label_col].dropna().astype(str).head(20).tolist()
                                sample_labels.append(f"Mẫu chỉ tiêu thực tế trong file '{Path(c_path).name}':\n{labels}")
                        except Exception:
                            pass
            sample_labels_str = "\n\n".join(sample_labels) if sample_labels else ""
            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu=muc_tieu,
                noi_dung=noi_dung,
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                sample_labels_str=sample_labels_str,
                previous_code=state.get('generated_code', ''),
                error_traceback=f"{error_traceback.strip()}\n\n{retry_forcing_msg}",
            )
            messages = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]

        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke(messages)
        raw_text = response.content if isinstance(response.content, str) else str(response.content)
        code = clean_python_code(raw_text)
        print(f"📊 [Kết quả - Code Generator] Mã Python sinh ra:\n```python\n{code}\n```\n")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)
        return {**state, "generated_code": code, "status": "pending", "node_latencies": node_latencies}
    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)
        return {**state, "status": "error", "error_message": f"Code generator error: {str(e)}", "node_latencies": node_latencies}

print("✅ Node 4: Code Generator loaded!")

### Node 5: AST Sandbox & Executor Node
Thực thi mã Python trong môi trường Sandbox AST an toàn và thu thập kết quả `result`.

In [14]:
import ast
import sys
import time
import json
import traceback
import pandas as pd
import numpy as np
from typing import Dict, Any, Optional

class SecurityError(Exception):
    """Raised when generated code contains forbidden AST nodes."""
    pass

FORBIDDEN_AST_NODES = (
    ast.Import,
    ast.ImportFrom,
)

FORBIDDEN_BUILTINS = {
    "eval", "exec", "__import__", "open", "compile",
    "globals", "locals", "input", "breakpoint"
}

ALLOWED_MODULES = {"pandas", "pd", "numpy", "np", "datetime", "math", "re"}

def validate_ast(code_str: str) -> None:
    """Validate Python code against AST safety rules."""
    tree = ast.parse(code_str)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_MODULES:
                    raise SecurityError(f"Importing forbidden module: '{alias.name}'")
        elif isinstance(node, ast.ImportFrom):
            if node.module and node.module.split(".")[0] not in ALLOWED_MODULES:
                raise SecurityError(f"Importing from forbidden module: '{node.module}'")
        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_BUILTINS:
                raise SecurityError(f"Call to forbidden function: '{node.func.id}'")

def format_result(result: Any) -> Any:
    """Format DataFrame, Series, or scalar result for JSON serialization."""
    if isinstance(result, pd.DataFrame):
        df_sub = result.head(100)
        return {
            "type": "dataframe",
            "shape": list(result.shape),
            "columns": list(result.columns),
            "data": df_sub.to_dict(orient="records"),
        }
    elif isinstance(result, pd.Series):
        s_sub = result.head(100)
        return {
            "type": "series",
            "name": str(result.name) if result.name else "result",
            "data": s_sub.to_dict(),
        }
    elif isinstance(result, (int, float, str, bool, list, dict)):
        return {
            "type": "scalar",
            "data": result,
        }
    else:
        return {
            "type": "other",
            "data": str(result),
        }

def executor_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()

    code_str = state.get("generated_code", "").strip()
    discovered_tables = state.get("discovered_tables", [])
    file_path = ""
    if discovered_tables:
        file_path = discovered_tables[0].get("csv_path", "")
    retry_count = state.get("retry_count", 0)

    if not code_str:
        return {
            **state,
            "status": "error",
            "error_traceback": "No code generated to execute.",
            "retry_count": retry_count + 1,
        }

    try:
        validate_ast(code_str)

        print(f"⚙️ [Executor] Đang thực thi mã Pandas...")

        df_loaded = None
        if file_path:
            try:
                df_loaded = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
            except Exception as e:
                print(f"⚠️ [Executor] Không thể tự động load DataFrame: {e}")

        # Robust clean_val definition (Vietnamese dash '-' means 0.0)
        def clean_val(val):
            if pd.isna(val):
                raise ValueError("Metric not found in table")
            val_str = str(val).strip()
            if val_str in ['-', '—']:
                return 0.0
            if not val_str or val_str in ['', 'nan', 'NaN', 'None', 'null', 'n/a']:
                raise ValueError("Metric not found in table")
            if isinstance(val, (int, float)): return float(val)
            neg = False
            if val_str.startswith('(') and val_str.endswith(')'):
                neg = True
                val_str = val_str[1:-1].strip()
            val_str = val_str.replace(',', '')
            if '.' in val_str:
                parts = val_str.split('.')
                if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) == 3):
                    val_str = val_str.replace('.', '')
            try:
                res = float(val_str)
                return -res if neg else res
            except Exception:
                raise ValueError("Metric not found in table")

        # Injected column scanning abstraction helper
        def extract_value(row, preferred_col):
            cols_to_try = [preferred_col, '1', '2', '3', '4', '5']
            for c in cols_to_try:
                if hasattr(row, 'index') and c in row.index:
                    try:
                        return clean_val(row[c])
                    except ValueError:
                        continue
                elif isinstance(row, pd.DataFrame) and c in row.columns and not row.empty:
                    try:
                        return clean_val(row[c].iloc[0])
                    except ValueError:
                        continue
            raise ValueError("Metric not found in table")

        exec_globals = {
            "pd": pd,
            "np": np,
            "pandas": pd,
            "numpy": np,
            "file_path": file_path,
            "df": df_loaded,
            "clean_val": clean_val,
            "extract_value": extract_value,
        }
        for tbl in discovered_tables:
            csv_p = tbl.get("csv_path", "")
            nam = tbl.get("Nam_Tai_Chinh", "")
            if csv_p:
                if nam:
                    exec_globals[f"file_path_{nam}"] = csv_p

        exec(code_str, exec_globals)

        result_val = exec_globals.get("result")

        if result_val is None:
            raise ValueError("Biến `result` không được tìm thấy sau khi thực thi mã.")

        formatted = format_result(result_val)
        
        print(f"✅ [Executor] Thực thi THÀNH CÔNG!")
        print(f"📊 [Kết quả - Executor]:\n{json.dumps(formatted, indent=4, ensure_ascii=False)}\n")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)

        return {
            **state,
            "execution_result": formatted,
            "error_traceback": None,
            "status": "success",
            "node_latencies": node_latencies,
        }

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)

        tb_str = traceback.format_exc()

        return {
            **state,
            "status": "error",
            "error_traceback": tb_str,
            "retry_count": retry_count + 1,
            "node_latencies": node_latencies,
        }

## 🌐 Section 5: LangGraph Workflow & Edge Routing
Xây dựng đồ thị trạng thái Agent StateGraph và thiết lập điều kiện Reflection Loop.

In [15]:
from langgraph.graph import StateGraph, END

def route_after_discovery(state: AgentState) -> Literal["schema_mapper", "__end__"]:
    """Route workflow after data discovery node."""
    if state.get("status") == "error":
        return END
    return "schema_mapper"

def route_after_schema_mapper(state: AgentState) -> Literal["code_generator", "__end__"]:
    """Route workflow after schema mapper node."""
    if state.get("status") == "error":
        return END
    return "code_generator"

def route_after_execution(state: AgentState, cfg: Optional[Config] = None) -> Literal["code_generator", "__end__"]:
    """Conditional edge for Reflection Debugging Loop."""
    cfg = cfg or config
    status = state.get("status")
    retry_count = state.get("retry_count", 0)

    if status == "success":
        return END

    if status == "error" and retry_count < cfg.MAX_RETRIES:
        print(f"🔄 Reflection Loop Activated! Retrying code generation ({retry_count}/{cfg.MAX_RETRIES})...")
        return "code_generator"

    return END

def create_cocopila_graph(cfg: Optional[Config] = None):
    """Construct and compile the LangGraph workflow graph with Schema Mapper."""
    cfg = cfg or config

    workflow = StateGraph(AgentState)

    # Add Nodes
    workflow.add_node("query_parser", lambda state: parse_query_node(state, cfg))
    workflow.add_node("data_discovery", lambda state: data_discovery_node(state, cfg))
    workflow.add_node("schema_mapper", lambda state: schema_mapper_node(state, cfg))
    workflow.add_node("code_generator", lambda state: code_generator_node(state, cfg))
    workflow.add_node("executor", lambda state: executor_node(state, cfg))

    # Define Workflow Edges
    workflow.set_entry_point("query_parser")
    workflow.add_edge("query_parser", "data_discovery")

    workflow.add_conditional_edges(
        "data_discovery",
        route_after_discovery,
        {
            "schema_mapper": "schema_mapper",
            END: END,
        }
    )

    workflow.add_conditional_edges(
        "schema_mapper",
        route_after_schema_mapper,
        {
            "code_generator": "code_generator",
            END: END,
        }
    )

    workflow.add_edge("code_generator", "executor")

    workflow.add_conditional_edges(
        "executor",
        lambda state: route_after_execution(state, cfg),
        {
            "code_generator": "code_generator",
            END: END,
        }
    )

    return workflow.compile()

print("✅ LangGraph Workflow Graph compiled with Schema Mapper integration!")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Đã khởi tạo thành công hàm create_cocopila_graph()!


## 🧪 Section 6: Dataset Linking & Running Agent Test

In [16]:
# 2. Dò tìm và liên kết trực tiếp Qdrant Local DB từ đường dẫn chỉ định trên Kaggle
import os
import shutil
from pathlib import Path

# Các đường dẫn khả thi trên Kaggle (bao gồm cả URL trình duyệt và đường dẫn hệ thống thực tế)
possible_dataset_dirs = [
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data"),
    Path("/kaggle/input/r2-ai-output/r2AI_data"),
    Path("/kaggle/input/r2-ai-output"),
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output"),
]

if Path("/kaggle/input").exists():
    print("🔍 Đang kiểm tra Kaggle Input Dataset cho Qdrant DB...")
    dataset_dir = None
    
    # 1. Thử các đường dẫn chỉ định trước
    for d in possible_dataset_dirs:
        if (d / "qdrant_local_db").exists():
            dataset_dir = d
            print(f"📦 Đã tìm thấy Qdrant DB tại đường dẫn chỉ định: {d / 'qdrant_local_db'}")
            break
            
    # 2. Nếu không thấy, dùng quét nông (shallow search) để tìm kiếm tự động
    if not dataset_dir:
        qdrant_found = []
        for depth_pattern in [
            "*/qdrant_local_db", 
            "*/*/qdrant_local_db", 
            "*/*/*/qdrant_local_db", 
            "*/*/*/*/qdrant_local_db", 
            "*/*/*/*/*/qdrant_local_db"
        ]:
            qdrant_found.extend(list(Path("/kaggle/input").glob(depth_pattern)))
            if qdrant_found:
                dataset_dir = qdrant_found[0].parent
                print(f"📦 Đã tìm thấy Qdrant DB bằng wildcard tại: {qdrant_found[0]}")
                break
            
    if dataset_dir:
        rag_module_dir = Path("rag_module").resolve()
        rag_module_dir.mkdir(exist_ok=True)
        
        # Symlink cho các file chỉ đọc (ViFinQA, bm25, code_stock)
        for item in ["bm25_index.pkl", "code_stock.csv", "ViFinQA"]:
            src = dataset_dir / item
            dst = rag_module_dir / item
            
            # Xóa liên kết cũ/hỏng nếu có để tránh lỗi FileExistsError
            if dst.exists() or dst.is_symlink():
                try:
                    if dst.is_symlink() or dst.is_file():
                        dst.unlink()
                    else:
                        shutil.rmtree(dst)
                except Exception as e:
                    print(f"   ⚠️ Cannot remove old {item}: {e}")
                    
            if src.exists() and not dst.exists():
                try:
                    os.symlink(src, dst)
                    print(f"   🔗 Created symlink: {dst} -> {src}")
                except Exception as e:
                    print(f"   ⚠️ Cannot symlink {item}: {e}")
        
        # Copy vật lý cho qdrant_local_db vì Qdrant yêu cầu quyền ghi (.lock file)
        qdrant_src = dataset_dir / "qdrant_local_db"
        qdrant_dst = rag_module_dir / "qdrant_local_db"
        
        # Xóa liên kết cũ/hỏng nếu có để tránh lỗi FileExistsError khi copytree
        if qdrant_dst.exists() or qdrant_dst.is_symlink():
            try:
                if qdrant_dst.is_symlink() or qdrant_dst.is_file():
                    qdrant_dst.unlink()
                else:
                    shutil.rmtree(qdrant_dst)
            except Exception as e:
                print(f"   ⚠️ Cannot remove old qdrant_local_db: {e}")
                
        if qdrant_src.exists() and not qdrant_dst.exists():
            print("   ⏳ Đang copy Qdrant DB sang thư mục làm việc để cấp quyền ghi (chỉ mất vài chục giây cho lần đầu)...")
            shutil.copytree(qdrant_src, qdrant_dst)
            print(f"   ✅ Đã copy xong Qdrant DB tới: {qdrant_dst}")
    else:
        print("❌ Không tìm thấy thư mục dataset trên Kaggle! Hãy kiểm tra xem bạn đã đính kèm dataset vào Notebook chưa.")
else:
    print("💻 Chạy local, sử dụng dữ liệu có sẵn tại rag_module/")

🔍 Đang kiểm tra Kaggle Input Dataset cho Qdrant DB...
📦 Đã tìm thấy Qdrant DB tại đường dẫn chỉ định: /kaggle/input/datasets/duymcminh/r2-ai-output/qdrant_local_db
   🔗 Created symlink: /kaggle/working/r2AI_2026/rag_module/bm25_index.pkl -> /kaggle/input/datasets/duymcminh/r2-ai-output/bm25_index.pkl
   🔗 Created symlink: /kaggle/working/r2AI_2026/rag_module/ViFinQA -> /kaggle/input/datasets/duymcminh/r2-ai-output/ViFinQA
   ⏳ Đang copy Qdrant DB sang thư mục làm việc để cấp quyền ghi (chỉ mất vài chục giây cho lần đầu)...
   ✅ Đã copy xong Qdrant DB tới: /kaggle/working/r2AI_2026/rag_module/qdrant_local_db


In [17]:
# 3. Khởi tạo Agent và thực thi 10 câu hỏi ngẫu nhiên (BOUNDED REFLECTION & ERROR HANDLING)
import time
import random
import json
from pathlib import Path
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

repo_dir = Path("/kaggle/working/r2AI_2026")
if not repo_dir.exists():
    repo_dir = Path.cwd()

# Nạp danh sách câu hỏi kiểm thử từ dataset
possible_qa_paths = [
    repo_dir / "rag_module" / "ViFinQA" / "questions" / "questions.jsonl",
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/rag_module/ViFinQA/questions/questions.jsonl"),
    Path("/kaggle/input/r2-ai-output/rag_module/ViFinQA/questions/questions.jsonl"),
    repo_dir / "rag_module" / "ViFinQA" / "ViFinQA_QA.json",
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data/ViFinQA/ViFinQA_QA.json"),
    Path("rag_module/ViFinQA/ViFinQA_QA.json"),
]

qa_file = None
for p in possible_qa_paths:
    if p.exists():
        qa_file = p
        break

qa_data = []
if qa_file and qa_file.exists():
    with open(qa_file, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if qa_file.suffix == ".jsonl" or "\n" in content:
            for line in content.splitlines():
                if line.strip():
                    try:
                        qa_data.append(json.loads(line))
                    except Exception:
                        pass
        else:
            try:
                qa_data = json.loads(content)
            except Exception:
                pass
    print(f"📋 Đã tải thành công {len(qa_data)} câu hỏi từ dataset!")
else:
    print("⚠️ Không tìm thấy file ViFinQA_QA.json, sử dụng câu hỏi mẫu mặc định.")
    qa_data = [
        {"id": 655, "question": "Tốc độ tăng trưởng % tổng tiền và các khoản tương đương tiền của VIC từ năm 2019 đến năm 2021 là bao nhiêu?"},
        {"id": 115, "question": "Tổng chi phí hoạt động của công ty mẹ Ngân hàng TMCP Công Thương Việt Nam năm 2019 là bao nhiêu triệu đồng?"},
        {"id": 26, "question": "Lãi vay phải trả của CTCP Tập đoàn Đức Long Gia Lai (DLG) cuối năm 2023 là bao nhiêu triệu đồng?"},
    ]

random.seed(67)
sample_questions = qa_data[:20] if len(qa_data) >= 20 else qa_data

results_summary = []
MAX_ATTEMPTS = 3

for idx, q_item in enumerate(sample_questions, 1):
    q_id = q_item.get("id", idx)
    q_text = q_item.get("question", "")
    print(f"\n[{idx}/{len(sample_questions)}] ❓ Câu hỏi ID {q_id}: {q_text}")
    print("-" * 60)
    
    try:
        # 1. STRICT STATELESS PARSER (Fixing Memory Leak)
        parser_prompt_data = PROMPT_QUERY_PARSER
        sys_prompt = parser_prompt_data.get("system_prompt", "")
        few_shots = parser_prompt_data.get("few_shot_examples", [])
        
        parser_messages = [SystemMessage(content=sys_prompt)]
        for ex in few_shots:
            parser_messages.append(HumanMessage(content=ex["user_query"]))
            parser_messages.append(SystemMessage(content=ex["parsed_output"]))
        parser_messages.append(HumanMessage(content=f"Câu hỏi: {q_text}"))
        
        llm_parser = get_llm(config, temperature=0.1, timeout=30)
        try:
            response_parser = llm_parser.invoke(parser_messages)
            raw_parser_out = response_parser.content if hasattr(response_parser, "content") else str(response_parser)
        except Exception as api_err:
            print(f"   ⏱️ LLM Parser API call failed or timed out: {api_err}")
            results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": "Parser timeout", "generated_code": "", "attempts": 0, "parsed_query": None, "discovered_tables": [], "result": None, "error_traceback": str(api_err) if "api_err" in locals() else None})
            continue

        parsed_json = safe_parse_json(raw_parser_out)
        
        ticker_val = parsed_json.get("ticker") or parsed_json.get("ten_cong_ty") or ""
        metric_val = parsed_json.get("metric") or parsed_json.get("noi_dung") or ""
        year_val = parsed_json.get("year") if "year" in parsed_json else parsed_json.get("so_nam")
        if not year_val or year_val is None or str(year_val).strip() in ["None", "null", ""]:
            year_val = re.findall(r"\b(20\d{2})\b", q_text)
        
        if isinstance(year_val, str):
            so_nam_list = [y.strip() for y in year_val.replace(",", " ").split() if y.strip().isdigit()]
            if not so_nam_list:
                so_nam_list = re.findall(r"\b(20\d{2})\b", q_text)
        elif isinstance(year_val, (int, float)):
            so_nam_list = [str(int(year_val))]
        elif isinstance(year_val, list):
            so_nam_list = [str(y).strip() for y in year_val if str(y).strip().isdigit()]
        else:
            so_nam_list = re.findall(r"\b(20\d{2})\b", q_text)
            
        company_clean = _normalize_company_name(ticker_val, q_text)
        parsed_json["ticker"] = ticker_val
        parsed_json["ten_cong_ty"] = company_clean
        parsed_json["year"] = ", ".join(so_nam_list) if so_nam_list else ""
        parsed_json["so_nam"] = so_nam_list
        parsed_json["metric"] = metric_val
        parsed_json["noi_dung"] = metric_val
        
        thao_tac = "so_sanh" if len(so_nam_list) > 1 or "so sánh" in q_text.lower() or "tăng trưởng" in q_text.lower() else "trich_xuat"
        parsed_json["thao_tac"] = thao_tac
        parsed_json["muc_tieu"] = thao_tac
        
        print(f"📊 [Query Parser Result]: Ticker={company_clean}, Year={so_nam_list}, Metric='{metric_val}'")
        
        state = {
            "user_query": q_text,
            "parsed_query": parsed_json,
            "discovered_tables": [],
            "column_mapping": {},
            "generated_code": "",
            "execution_result": None,
            "error_traceback": None,
            "retry_count": 0,
            "status": "pending",
            "error_message": None,
            "node_latencies": {},
        }
        
        state = data_discovery_node(state, config)
        if state.get("status") == "error":
            print(f"   ❌ Data Discovery Failed: {state.get('error_message')}")
            results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": state.get("error_message"), "generated_code": "", "attempts": 0, "parsed_query": parsed_json, "discovered_tables": [], "result": None, "error_traceback": None})
            continue

        # 1.5 SCHEMA MAPPER NODE (Extract useful_columns & sub_sections with LLM description)
        state = schema_mapper_node(state, config)
        schema = state.get("schema", {})
        print(f"📊 [Schema Mapper Summary]: Useful Cols={len(schema.get('useful_columns', []))}, Sub-sections={len(schema.get('sub_sections', []))}")
            
        # 2. CODE GEN & REFLECTION LOOP (Bounded Attempts & Standardized Exceptions)
        discovered_tables = state.get("discovered_tables", [])
        column_mapping = state.get("column_mapping", {})
        table_schema = state.get("table_schema", [])
        first_row_values = state.get("first_row_values", {})
        label_col = _resolve_label_column(table_schema, first_row_values, column_mapping)
        value_col = _resolve_value_column(table_schema, first_row_values, parsed_json, column_mapping, label_col, schema=schema)
        files_context = _build_files_context(discovered_tables, column_mapping, table_schema, first_row_values, schema=schema)
        
        paths_str = ""
        if discovered_tables:
            if len(discovered_tables) == 1:
                escaped = discovered_tables[0]["csv_path"].replace('\\', '/')
                paths_str = f"file_path = '{escaped}'"
            else:
                for tbl in discovered_tables:
                    nam = tbl.get("Nam_Tai_Chinh", "default")
                    escaped = tbl["csv_path"].replace('\\', '/')
                    paths_str += f"file_path_{nam} = '{escaped}'\n"
        
        cg_prompt_data = PROMPT_CODE_GENERATOR
        cg_sys_prompt = cg_prompt_data.get("system_prompt", "")
        goal_desc = cg_prompt_data.get("goal_descriptions", {}).get(thao_tac, thao_tac)
        goal_inst_template = cg_prompt_data.get("goal_instructions", {}).get(thao_tac, "")
        goal_inst = goal_inst_template.format(noi_dung=metric_val, label_col=label_col, value_col=value_col) if goal_inst_template else ""
        
        user_template = cg_prompt_data.get("user_prompt_template", "")
        human_init_content = user_template.format(
            user_query=q_text,
            muc_tieu_desc=goal_desc,
            noi_dung=metric_val,
            ten_cong_ty=company_clean,
            so_nam=so_nam_list,
            tieu_chi_phu="(không có)",
            files_context=files_context,
            label_col=label_col,
            value_col=value_col,
            paths_str=paths_str,
            goal_instruction=goal_inst,
        )
        
        code_messages = [
            SystemMessage(content=cg_sys_prompt),
            HumanMessage(content=human_init_content)
        ]
        
        llm_codegen = get_llm(config, temperature=0.0, timeout=30)
        execution_success = False
        
        for attempt in range(MAX_ATTEMPTS):
            print(f"⚙️ [Code Generator] Generation attempt {attempt + 1}/{MAX_ATTEMPTS}...")
            try:
                response_code = llm_codegen.invoke(code_messages)
                raw_code = response_code.content if hasattr(response_code, "content") else str(response_code)
            except Exception as api_err:
                print(f"   ⏱️ LLM CodeGen API call failed or timed out on attempt {attempt + 1}: {api_err}")
                break
                
            code = clean_python_code(raw_code)
            state["generated_code"] = code
            
            # Execute code inside AST Sandbox safely (catches all exceptions including TypeError and ValueError)
            try:
                exec_output = executor_node(state, config)
            except Exception as exec_err:
                exec_output = {"status": "error", "error_traceback": f"Execution error: {str(exec_err)}"}
                
            if exec_output.get("status") == "success":
                print(f"✅ [Code Generator] Execution SUCCESS on attempt {attempt + 1}!")
                execution_success = True
                results_summary.append({"id": q_id, "question": q_text, "status": "success", "result": exec_output.get("execution_result"), "generated_code": code, "attempts": attempt + 1, "parsed_query": parsed_json, "discovered_tables": discovered_tables, "error_traceback": None})
                break
            else:
                err_msg = exec_output.get("error_traceback", "Execution failed")
                last_err_line = err_msg.splitlines()[-1] if err_msg else "Execution failed"
                print(f"⚠️ Attempt {attempt + 1}/{MAX_ATTEMPTS} failed: {last_err_line}")
                
                # Check if this was the last attempt: DO NOT append new messages or continue retry loop
                if attempt >= MAX_ATTEMPTS - 1:
                    print(f"🛑 Reached maximum attempts ({MAX_ATTEMPTS}). Gracefully terminating reflection loop for Question ID {q_id}.")
                    break
                
                # IMPROVED REFLECTION PROMPT
                error_instruction = (
                    f"Execution Failed: {last_err_line}."
                    "If your previous attempt failed because the row was not found, DO NOT write the exact same string matching code again. "
                    "You must try matching a different variation of the row name, or use partial string matching (str.contains)."
                    "CRITICAL: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'."
                )
                
                # Appends previous assistant response and retry human prompt safely
                code_messages.append(AIMessage(content=f"```python\n{code}\n```"))
                code_messages.append(HumanMessage(content=error_instruction))
        
        if not execution_success:
            print(f"❌ Question ID {q_id} failed after reflection/retry attempts.")
            results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": "Failed after reflection attempts", "result": {"type": "error", "data": None}, "generated_code": state.get("generated_code", ""), "attempts": MAX_ATTEMPTS, "parsed_query": parsed_json, "discovered_tables": discovered_tables, "error_traceback": exec_output.get("error_traceback") if "exec_output" in locals() else None})

    except Exception as e:
        print(f"   💥 Lỗi ngoại lệ trong quá trình chạy: {e}")
        results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": str(e), "result": {"type": "error", "data": None}, "generated_code": state.get("generated_code", "") if "state" in locals() else "", "attempts": attempt + 1 if "attempt" in locals() else 0, "parsed_query": parsed_json if "parsed_json" in locals() else None, "discovered_tables": state.get("discovered_tables", []) if "state" in locals() else [], "error_traceback": str(e)})

print("\n" + "=" * 80)
print("📊 TỔNG HỢP KẾT QUẢ KIỂM THỬ:")
print("=" * 80)
success_count = sum(1 for r in results_summary if r["status"] == "success")
print(f"Tổng số câu hỏi: {len(results_summary)} | Thành công: {success_count} | Thất bại: {len(results_summary) - success_count}")
for r in results_summary:
    status_emoji = "✅" if r["status"] == "success" else "❌"
    print(f"{status_emoji} ID {r['id']}: {r['status'].upper()}")

📋 Đã tải thành công 1012 câu hỏi từ dataset!

[1/20] ❓ Câu hỏi ID 1: Lãi tiền gửi năm 2018 của công ty mẹ CTCP Hàng không Vietjet (VJC) là bao nhiêu triệu đồng?
------------------------------------------------------------
📊 [Query Parser Result]: Ticker=VJC, Year=['2018'], Metric='Lãi tiền gửi'

🔍 [Data Discovery] Bắt đầu tìm kiếm dữ liệu...
   - Công ty: 'VJC', Số năm: ['2018'], Nội dung đã làm sạch: 'Lãi tiền gửi' (gốc: 'Lãi tiền gửi')


Directory /kaggle/input/datasets/duymcminh/r2-ai-output/qdrant_local_db is read-only. Copying database to writable location: /tmp/qdrant_db_qdrant_local_db
/kaggle/working/r2AI_2026/rag_module/search_engine.py:182: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <financial_tables> contains 451386 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  return QdrantClient(path=str(test_path))


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   📋 Danh sách 5 bảng ứng viên Top-K từ Search Engine (Năm 2018):
      #1 RRF: 0.132522 | File: VJC_financial_statements_2018_separate_table_49_0.csv | Hàng khớp: 'Lãi tiền gửi' | Tên bảng: 29. Doanh thu hoạt động tài chính
      #2 RRF: 0.132522 | File: VJC_financial_statements_2018_separate_table_49_0@line_1179.csv | Hàng khớp: 'Lãi tiền gửi' | Tên bảng: 29. Doanh thu hoạt động tài chính
      #3 RRF: 0.131746 | File: VJC_financial_statements_2018_separate_table_49.csv | Hàng khớp: 'Lãi tiền gửi' | Tên bảng: 29. Doanh thu hoạt động tài chính
      #4 RRF: 0.128850 | File: VJC_financial_statements_2018_consolidated_table_13.csv | Hàng khớp: 'Lãi tiền gửi và cho vay' | Tên bảng: Báo cáo lưu chuyển tiền tệ hợp nhất cho năm kết thúc ngày 31 tháng 12 năm 2018 Phương pháp gián tiếp
      #5 RRF: 0.128571 | File: VJC_financial_statements_2018_consolidated_table_56.csv | Hàng khớp: 'Lãi tiền gửi và cho vay' | Tên bảng: 29. Giá vốn hàng bán và cung cấp dịch vụ 30. Doanh thu hoạt động tài chí

## 💾 Section 7: Lưu trữ & Kiểm thử thủ công kết quả Code Generator (Export, Zip & Manual Verification)
Mục này cung cấp các công cụ hỗ trợ kiểm thử thủ công và debug:
1. **Lưu trữ kết quả**: Xuất toàn bộ kết quả chạy kèm mã nguồn Python (`generated_code`) ra file JSON và CSV.
2. **Xuất Scripts riêng biệt**: Tự động sinh file script `.py` cho từng câu hỏi vào thư mục `manual_test_scripts/` để kiểm tra độc lập.
3. **Nén ZIP tự động**: Đóng gói toàn bộ output (`.json`, `.csv`, `.py` scripts) thành một file `.zip` duy nhất (`codegen_manual_tests.zip`) để dễ dàng tải về từ Kaggle/Local.
4. **Bảng tổng hợp trực quan**: DataFrame hiển thị trạng thái, kết quả và độ dài mã của từng câu hỏi.
5. **Hàm kiểm thử thủ công `inspect_and_run(q_id)`**: Cho phép in toàn bộ mã nguồn sinh ra, xem các bảng dữ liệu liên quan và re-run trực tiếp mã nguồn trong Sandbox AST.

In [18]:
# 1. Khởi tạo thư mục lưu trữ kết quả kiểm thử
import json
import shutil
import zipfile
import pandas as pd
from pathlib import Path

output_dir = Path("/kaggle/working/manual_test_scripts") if Path("/kaggle/working").exists() else Path("./manual_test_scripts")
output_dir.mkdir(parents=True, exist_ok=True)

results_file_json = output_dir / "codegen_results.json"
results_file_csv = output_dir / "codegen_results.csv"

# 2. Lưu toàn bộ kết quả vào file JSON (đầy đủ metadata, code, tracebacks)
with open(results_file_json, "w", encoding="utf-8") as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

# 3. Xuất từng đoạn code sinh ra thành các file .py độc lập kèm header metadata
for item in results_summary:
    q_id = item.get("id")
    status = item.get("status", "unknown")
    code = item.get("generated_code", "")
    question = item.get("question", "")
    result = item.get("result")
    err = item.get("error")
    attempts = item.get("attempts", 1)
    
    script_path = output_dir / f"test_q_{q_id}_{status}.py"
    header_comment = f'''"""
======================================================================
QUESTION ID : {q_id}
STATUS      : {status} (Attempts: {attempts})
QUESTION    : {question}
RESULT      : {json.dumps(result, ensure_ascii=False) if result else 'None'}
ERROR       : {err if err else 'None'}
======================================================================
"""
'''
    with open(script_path, "w", encoding="utf-8") as sf:
        sf.write(header_comment + "\n" + (code if code else "# Không có mã nguồn Python nào được sinh ra."))

# 4. Sao chép file log .txt vào thư mục kết quả để đóng gói
log_file_dest = output_dir / "pipeline_execution.txt"
if LOG_FILE_PATH.exists():
    shutil.copy(LOG_FILE_PATH, log_file_dest)

# 5. Tạo bảng DataFrame tổng hợp kết quả
summary_rows = []
for item in results_summary:
    res = item.get("result")
    res_val = res.get("data") if isinstance(res, dict) else res
    res_type = res.get("type") if isinstance(res, dict) else type(res).__name__
    code_text = item.get("generated_code", "")
    
    summary_rows.append({
        "ID": item.get("id"),
        "Câu hỏi": item.get("question", "")[:60] + ("..." if len(item.get("question", "")) > 60 else ""),
        "Trạng thái": "✅ SUCCESS" if item.get("status") == "success" else "❌ ERROR",
        "Số lần thử": item.get("attempts", 1),
        "Kết quả": str(res_val)[:40] if res_val is not None else "None",
        "Kiểu dữ liệu": res_type,
        "Lỗi": item.get("error") or "Không có",
        "Độ dài mã (dòng)": len(code_text.splitlines()) if code_text else 0,
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(results_file_csv, index=False, encoding="utf-8-sig")

# 6. Nén toàn bộ thư mục output (bao gồm file log .txt) thành 1 file ZIP duy nhất
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
zip_file_path = (working_dir / "codegen_manual_tests.zip").resolve()

with zipfile.ZipFile(zip_file_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(output_dir.rglob("*")):
        if file.is_file():
            zf.write(file, arcname=file.name)

zip_size_kb = round(zip_file_path.stat().st_size / 1024, 2) if zip_file_path.exists() else 0
log_size_kb = round(LOG_FILE_PATH.stat().st_size / 1024, 2) if LOG_FILE_PATH.exists() else 0

print("=" * 80)
print("💾 ĐÃ LƯU & NÉN TOÀN BỘ KẾT QUẢ KIỂM THỬ THÀNH CÔNG!")
print("=" * 80)
print(f"📁 Thư mục mã nguồn kiểm thử : {output_dir.resolve()}")
print(f"📝 File log toàn bộ quá trình: {LOG_FILE_PATH.resolve()} ({log_size_kb} KB)")
print(f"📄 File kết quả JSON         : {results_file_json.resolve()}")
print(f"📄 File kết quả CSV          : {results_file_csv.resolve()}")
print(f"📦 FILE ZIP TỔNG HỢP OUTPUT  : {zip_file_path} ({zip_size_kb} KB)")
print("=" * 80)

# Hiển thị bảng tổng hợp
display(df_summary)

# 7. Hàm tiện ích để kiểm thử thủ công & Re-run từng câu hỏi
def inspect_and_run(target_q_id: int):
    """Xem chi tiết câu hỏi, mã nguồn Python sinh ra và thực thi lại trong Sandbox AST."""
    matched = [r for r in results_summary if r.get("id") == target_q_id]
    if not matched:
        print(f"❌ Không tìm thấy câu hỏi với ID = {target_q_id} trong kết quả kiểm thử.")
        return
    
    item = matched[0]
    print("\n" + "=" * 80)
    print(f"🔍 CHI TIẾT KIỂM THỬ THỦ CÔNG - CÂU HỎI ID: {target_q_id}")
    print("=" * 80)
    print(f"❓ Câu hỏi    : {item.get('question')}")
    print(f"📊 Trạng thái : {item.get('status')}")
    print(f"🔄 Số lần thử : {item.get('attempts', 1)}")
    print(f"🎯 Kết quả cũ : {item.get('result')}")
    if item.get("error"):
        print(f"⚠️ Thông báo lỗi: {item.get('error')}")
    
    tables = item.get("discovered_tables", [])
    if tables:
        print(f"\n📂 Các bảng dữ liệu liên quan ({len(tables)} bảng):")
        for t in tables:
            print(f"   - Năm {t.get('Nam_Tai_Chinh', 'N/A')}: {t.get('csv_path')}")
            
    code = item.get("generated_code", "")
    print("\n💻 MÃ NGUỒN PYTHON ĐƯỢC SINH RA:")
    print("-" * 80)
    if code:
        print(code)
    else:
        print("# (Không có mã nguồn nào được sinh ra)")
        print("-" * 80)
        return
    print("-" * 80)
    
    print("\n⚡ TIẾN HÀNH THỰC THI LẠI TRONG SANDBOX (RE-RUN):")
    re_state = {
        "user_query": item.get("question", ""),
        "generated_code": code,
        "discovered_tables": tables,
        "status": "pending",
        "error_traceback": None,
        "execution_result": None
    }
    try:
        re_output = executor_node(re_state, config)
        if re_output.get("status") == "success":
            print(f"👉 Re-run THÀNH CÔNG! Kết quả: {re_output.get('execution_result')}")
        else:
            print(f"👉 Re-run THẤT BẠI!")
            print(f"⚠️ Traceback: {re_output.get('error_traceback')}")
    except Exception as re_err:
        print(f"💥 Ngoại lệ khi re-run: {re_err}")

print("\n💡 HƯỚNG DẪN KIỂM THỬ THỦ CÔNG:")
print("   Để xem mã nguồn và chạy thử lại bất kỳ câu hỏi nào, gọi hàm:")
first_id = results_summary[0]['id'] if results_summary else 1
print(f"   >>> inspect_and_run({first_id})")

💾 ĐÃ LƯU & NÉN TOÀN BỘ KẾT QUẢ KIỂM THỬ THỦ CÔNG THÀNH CÔNG!
📁 Thư mục mã nguồn kiểm thử : /kaggle/working/manual_test_scripts
📄 File kết quả JSON         : /kaggle/working/manual_test_scripts/codegen_results.json
📄 File kết quả CSV          : /kaggle/working/manual_test_scripts/codegen_results.csv
📦 FILE ZIP TỔNG HỢP OUTPUT  : /kaggle/working/codegen_manual_tests.zip (20.48 KB)


,ID,Câu hỏi,Trạng thái,Số lần thử,Kết quả,Kiểu dữ liệu,Lỗi,Độ dài mã (dòng)
0,1,Lãi tiền gửi năm 2018 của công ty mẹ CTCP Hàng...,✅ SUCCESS,1,208253201298.0,scalar,Không có,10
1,2,Số dư cho vay khách hàng ngành Thương mại của ...,❌ ERROR,3,None,error,Failed after reflection attempts,10
2,3,Chi phí dự phòng của Ngân hàng TMCP Sài Gòn Tà...,✅ SUCCESS,1,1422948.0,scalar,Không có,10
3,4,Lợi nhuận sau thuế của CTCP Chứng khoán FPT nă...,✅ SUCCESS,1,421.0,scalar,Không có,10
4,5,Chi phí phạt của công ty mẹ SCR năm 2017 là ba...,✅ SUCCESS,1,5535987434.0,scalar,Không có,10
5,6,Lưu chuyển tiền thuần từ hoạt động kinh doanh ...,✅ SUCCESS,1,40.0,scalar,Không có,10
6,7,"Quỹ khen thưởng, phúc lợi của HT1 cuối năm 201...",❌ ERROR,3,None,error,Failed after reflection attempts,10
7,8,Chi phí lương và các khoản khác theo lương của...,❌ ERROR,3,None,error,Failed after reflection attempts,10
8,9,Chi phí khác của SAM năm 2023 là bao nhiêu tri...,✅ SUCCESS,1,2307758675.0,scalar,Không có,10
9,10,Chi phí tài chính của công ty mẹ CTCP Phát tri...,✅ SUCCESS,1,808762625018.0,scalar,Không có,10



💡 HƯỚNG DẪN KIỂM THỬ THỦ CÔNG:
   Để xem mã nguồn và chạy thử lại bất kỳ câu hỏi nào, gọi hàm:
   >>> inspect_and_run(1)
